In [ ]:
# --- portable setup (added 2026-09-12 when the project moved to GitHub) ---------------------
# All paths below resolve from HME_ROOT, the repository root.  On Colab: mount Drive and point
# HME_ROOT at your clone.  Locally: run jupyter from the repo, or export HME_ROOT=/path/to/repo.
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    HME_ROOT = os.environ.get('HME_ROOT', '/content/drive/MyDrive/hyperbolic-icd10')   # <-- edit
except ImportError:
    HME_ROOT = os.environ.get('HME_ROOT', os.path.abspath(os.path.join(os.getcwd(), '..')))
assert os.path.isdir(os.path.join(HME_ROOT, 'data')), f'HME_ROOT={HME_ROOT!r} is not the repo root'
print('HME_ROOT =', HME_ROOT)


# Sandbox — Fast Tweak Testing

Quick experiments to hunt for MAP gains. Designed for SPEED: d=10, short epochs, ~1 min per test.

**Anti-noise discipline:** every test prints the baseline next to it, and there's a `multi_seed` helper so you can check whether a gain is real (> ~0.003) or seed noise (±0.0015). Don't trust a single-seed +0.002.

**Setup once (cells 1-4), then each test cell is independent — run any, in any order.**

Tests included:
- T1: alpha sweep ON THE GRADED LAW (never optimized — you tuned alpha for binary)
- T2: kappa_scale / bandwidth on graded
- T3: epochs & lr (are we under-training?)
- T4: negatives count
- T5: WARM-START the learnable field at graded (the real shot — learn beyond the law)
- T6: multi-seed any promising config to confirm it's real

## 1. Setup + load

In [1]:
!pip install geoopt
import os, pickle, numpy as np, torch, torch.nn as nn, geoopt
from collections import defaultdict, deque
ROOT_DIR=HME_ROOT
DATA_DIR=os.path.join(ROOT_DIR,'data/processed')
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device',device)
with open(os.path.join(DATA_DIR,'icd10_tree_with_features.pkl'),'rb') as f: data=pickle.load(f)
nodes=data['nodes']; edges=data['edges']; features=data['features']
codes=list(nodes.keys()); code_to_idx={c:i for i,c in enumerate(codes)}; idx_to_code={i:c for c,i in code_to_idx.items()}
N=len(codes); edges_idx=[(code_to_idx[p],code_to_idx[c]) for p,c in edges]
kids_idx=defaultdict(list)
for u,v in edges_idx: kids_idx[u].append(v)
ROOT=code_to_idx['ROOT']; connected=set(edges_idx)|set((v,u) for u,v in edges_idx)
print(f'N={N}')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1/90.1 kB 9.8 MB/s eta 0:00:00
device cuda
N=46817


## 2. Fast eval (smaller sample for speed)

In [2]:
nbrs=defaultdict(set)
for u,v in edges_idx: nbrs[u].add(v); nbrs[v].add(u)
rng=np.random.default_rng(42)
EVAL=[edges_idx[i] for i in rng.choice(len(edges_idx),size=1000,replace=False)]  # 1000 for speed
def d_poincare(a,allp):
    diff2=np.sum((allp-a)**2,axis=1); nu=1-np.sum(a**2); nv=1-np.sum(allp**2,axis=1)
    return np.arccosh(np.maximum(1+2*diff2/(nu*nv+1e-12),1.0))
def evaluate(pos):
    if np.isnan(pos).any(): return float('nan'),float('nan')
    ranks=[]; aps=[]
    for (u,v) in EVAL:
        d=d_poincare(pos[u],pos); d[u]=np.inf; order=np.argsort(d)
        ranks.append(int(np.where(order==v)[0][0])+1)
        truth=nbrs[u]; hit=0; precs=[]
        for j,node in enumerate(order):
            if node in truth: hit+=1; precs.append(hit/(j+1))
            if hit==len(truth): break
        if precs: aps.append(np.mean(precs))
    return float(np.mean(aps)),float(np.mean(ranks))
print('fast eval ready (1000 edges)')

fast eval ready (1000 edges)


## 3. Phase 2 Poincaré components (verbatim)

In [3]:
class PoincareEmbedding(nn.Module):
    """
    Poincaré ball embedding model with constant curvature K = -1.

    Stores N embedding vectors that live inside the open unit ball in d dimensions.
    The Poincaré metric makes distances explode near the unit sphere, which gives
    hyperbolic space its exponential volume growth — the property that lets
    low-dimensional embeddings capture tree structure efficiently.
    """

    def __init__(self, num_nodes, dim, init_scale=0.001):
        super().__init__()

        # The Poincaré ball manifold from geoopt.
        # The 'c' parameter is the absolute value of curvature: c=1 means K=-1.
        # Internally, geoopt uses negative-curvature conventions consistent with
        # Nickel-Kiela's original paper.
        self.manifold = geoopt.PoincareBall(c=1.0)

        # The embedding table, but as a ManifoldParameter that geoopt knows
        # is constrained to live on the Poincaré ball manifold. This is what
        # tells the Riemannian optimizer to respect the unit-ball constraint.
        self.embeddings = geoopt.ManifoldParameter(
            torch.empty(num_nodes, dim),
            manifold=self.manifold
        )

        # Initialize embeddings to small random values near the origin.
        # Near-origin initialization is critical for hyperbolic embeddings:
        # the metric becomes singular at the boundary, so starting close to
        # the boundary causes numerical instability.
        # The .data accessor lets us modify the parameter in-place without
        # tracking gradients.
        with torch.no_grad():
            self.embeddings.data.uniform_(-init_scale, init_scale)

        self.dim = dim
        self.num_nodes = num_nodes

    def forward(self, indices):
        """
        Look up embeddings for a batch of indices.

        Parameters
        ----------
        indices : torch.LongTensor of any shape
            Integer indices into the embedding table.

        Returns
        -------
        embeddings : torch.FloatTensor
            One trailing dimension of size self.dim added to the input shape.
        """
        # Index into the embedding table along axis 0
        return self.embeddings[indices]

    def distance(self, u, v):
        """
        Poincaré distance between two batches of embeddings.

        Parameters
        ----------
        u, v : torch.FloatTensor of shape [..., dim]
            Two batches of points in the Poincaré ball.

        Returns
        -------
        d : torch.FloatTensor of shape [...]
            Hyperbolic distance, one per pair.
        """
        # geoopt's manifold object provides the correct distance function.
        # Internally this computes d(u,v) = arcosh(1 + 2||u-v||^2 / [(1-||u||^2)(1-||v||^2)])
        return self.manifold.dist(u, v, keepdim=False)

def compute_loss_with_hierarchy(
    anchor_emb, positive_emb, negative_embs, distance_fn,
    margin=0.05, lambda_h=1.0,
):
    """Nickel-Kiela softmax loss + hierarchy regularization."""
    # Standard NK loss
    pos_dist = distance_fn(anchor_emb, positive_emb)
    neg_dist = distance_fn(anchor_emb.unsqueeze(1), negative_embs)
    all_dist = torch.cat([pos_dist.unsqueeze(1), neg_dist], dim=1)
    logsumexp_term = torch.logsumexp(-all_dist, dim=1)
    nk_per_anchor = pos_dist + logsumexp_term

    # Hierarchy regularization: penalize parent norm > child norm - margin
    anchor_norms = torch.norm(anchor_emb, dim=-1)
    positive_norms = torch.norm(positive_emb, dim=-1)
    hierarchy_violation = torch.relu(anchor_norms - positive_norms + margin)

    return nk_per_anchor.mean() + lambda_h * hierarchy_violation.mean()

def train_one_epoch_with_freeze_and_hierarchy(
    model, edges_idx, optimizer, batch_size, num_negatives,
    N, connected, device,
    margin=0.05, lambda_h=1.0,
    freeze_indices=None,
):
    """One training epoch with optional gradient-zero freeze on specific rows."""
    model.train()
    shuffled_edges = list(edges_idx)
    np.random.shuffle(shuffled_edges)

    total_loss = 0.0
    for batch_start in tqdm(range(0, len(shuffled_edges), batch_size),
                             desc="Training", leave=False):
        batch_edges = shuffled_edges[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_edges)

        anchor_indices = np.array([e[0] for e in batch_edges], dtype=np.int64)
        positive_indices = np.array([e[1] for e in batch_edges], dtype=np.int64)
        negative_indices = sample_negatives_batched(
            anchor_indices, num_negatives, N, connected,
        )

        anchor_t = torch.from_numpy(anchor_indices).to(device)
        positive_t = torch.from_numpy(positive_indices).to(device)
        negative_t = torch.from_numpy(negative_indices).to(device)

        anchor_emb = model(anchor_t)
        positive_emb = model(positive_t)
        negative_embs = model(negative_t)

        loss = compute_loss_with_hierarchy(
            anchor_emb, positive_emb, negative_embs,
            model.distance,
            margin=margin, lambda_h=lambda_h,
        )

        optimizer.zero_grad()
        loss.backward()

        # Zero out gradient rows for frozen indices (e.g., ROOT)
        if freeze_indices is not None and model.embeddings.grad is not None:
            for idx in freeze_indices:
                model.embeddings.grad[idx].zero_()

        optimizer.step()
        total_loss += loss.item() * actual_batch_size

    return total_loss / len(shuffled_edges)

def train_one_epoch_with_freeze_and_hierarchy(
    model, edges_idx, optimizer, batch_size, num_negatives,
    N, connected, device,
    margin=0.05, lambda_h=1.0,
    freeze_indices=None,
):
    """train_one_epoch_with_freeze, but using compute_loss_with_hierarchy."""
    model.train()
    shuffled_edges = list(edges_idx)
    np.random.shuffle(shuffled_edges)

    total_loss = 0.0
    for batch_start in tqdm(range(0, len(shuffled_edges), batch_size),
                             desc="Training", leave=False):
        batch_edges = shuffled_edges[batch_start : batch_start + batch_size]
        actual_batch_size = len(batch_edges)

        anchor_indices = np.array([e[0] for e in batch_edges], dtype=np.int64)
        positive_indices = np.array([e[1] for e in batch_edges], dtype=np.int64)
        negative_indices = sample_negatives_batched(
            anchor_indices, num_negatives, N, connected,
        )

        anchor_t = torch.from_numpy(anchor_indices).to(device)
        positive_t = torch.from_numpy(positive_indices).to(device)
        negative_t = torch.from_numpy(negative_indices).to(device)

        anchor_emb = model(anchor_t)
        positive_emb = model(positive_t)
        negative_embs = model(negative_t)

        loss = compute_loss_with_hierarchy(
            anchor_emb, positive_emb, negative_embs,
            model.distance,
            margin=margin, lambda_h=lambda_h,
        )

        optimizer.zero_grad()
        loss.backward()

        if freeze_indices is not None and model.embeddings.grad is not None:
            for idx in freeze_indices:
                model.embeddings.grad[idx].zero_()

        optimizer.step()
        total_loss += loss.item() * actual_batch_size

    return total_loss / len(shuffled_edges)


def train_poincare_with_freeze_and_hierarchy(
    edges_idx, N, connected, code_to_idx,
    dim=10,
    num_epochs=300,
    burnin_epochs=10,
    freeze_train_epochs=10000,
    batch_size=1024,
    num_negatives=50,
    learning_rate=10.0,
    burnin_multiplier=0.01,
    init_scale=0.001,
    margin=0.05,
    lambda_h=1.0,
    device=None,
):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print("=" * 60)
    print(f"🔒+🏛️ ROOT-FREEZE + HIERARCHY Poincaré training")
    print(f"   ROOT idx = {code_to_idx['ROOT']}, pinned for {burnin_epochs + freeze_train_epochs} epochs")
    print(f"   hierarchy: margin={margin}, lambda_h={lambda_h}")
    print(f"   lr={learning_rate}, num_negatives={num_negatives}")
    print("=" * 60)

    model = PoincareEmbedding(num_nodes=N, dim=dim, init_scale=init_scale).to(device)

    root_idx = code_to_idx['ROOT']
    with torch.no_grad():
        model.embeddings.data[root_idx].zero_()

    with torch.no_grad():
        root_norm_at_init = torch.norm(model.embeddings.data[root_idx]).item()
        print(f"After init: ROOT_norm = {root_norm_at_init:.6f}   (must be 0.000000)")

    optimizer = geoopt.optim.RiemannianSGD(model.parameters(), lr=learning_rate)

    loss_history = []
    freeze_until_epoch = burnin_epochs + freeze_train_epochs

    for epoch in range(num_epochs):
        effective_lr = (learning_rate * burnin_multiplier
                        if epoch < burnin_epochs else learning_rate)
        for pg in optimizer.param_groups:
            pg['lr'] = effective_lr

        freeze_indices = [root_idx] if epoch < freeze_until_epoch else None

        avg_loss = train_one_epoch_with_freeze_and_hierarchy(
            model, edges_idx, optimizer,
            batch_size, num_negatives, N, connected, device,
            margin=margin, lambda_h=lambda_h,
            freeze_indices=freeze_indices,
        )
        loss_history.append(avg_loss)

        if epoch < burnin_epochs:
            phase = "burn-in+freeze"
        elif epoch < freeze_until_epoch:
            phase = "FROZEN+hier"
        else:
            phase = "free+hier"

        if (epoch < 3 or epoch == burnin_epochs or epoch == freeze_until_epoch
                or (epoch + 1) % 10 == 0):
            with torch.no_grad():
                norms = torch.norm(model.embeddings.data, dim=-1)
                print(
                    f"Epoch {epoch+1:3d}/{num_epochs} [{phase}, lr={effective_lr:.4f}]: "
                    f"loss={avg_loss:.4f}, "
                    f"ROOT_norm={norms[code_to_idx['ROOT']].item():.4f}, "
                    f"mean_norm={norms.mean().item():.4f}, "
                    f"max_norm={norms.max().item():.4f}"
                )

    return model, loss_history



## 4. Flexible trainer + kappa laws

In [4]:
bf_all=np.array([features[idx_to_code[i]]['branching_factor'] for i in range(N)])
kr=np.zeros((N,2),dtype=np.float32)
for cs,i in code_to_idx.items():
    f=features[cs]; kr[i,0]=np.log1p(f['branching_factor'])**2; kr[i,1]=np.log1p(f['subtree_size'])
kappa_raw=torch.tensor(kr,device=device)

def graded_kappa(scale=1.0):
    k=np.log1p(bf_all)**2; k=2*(k-k.min())/(k.max()-k.min()+1e-9)-1
    return torch.tensor(k*scale,dtype=torch.float32,device=device)

def sample_negs(anchors,K):
    out=np.random.randint(0,N,size=(len(anchors),K))
    for i,a in enumerate(anchors):
        for j in range(K):
            while out[i,j]==a or (a,out[i,j]) in connected: out[i,j]=np.random.randint(0,N)
    return out

def run(mode='graded', dim=10, alpha=-0.8, epochs=100, K=50, lr=10.0,
        burnin=10, burnin_mult=0.01, margin=0.05, lambda_h=1.0,
        kappa_scale=1.0, warmstart_graded=False, seed=0, verbose=False):
    """mode: const | learned | graded. Returns (MAP, mean_rank)."""
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    W=None; fopt=None
    if mode=='learned':
        W=nn.Parameter(torch.zeros(2,1,device=device))
        if warmstart_graded:
            # REAL warm-start: pre-fit W to match the graded law, in a SEPARATE graph
            target=graded_kappa(kappa_scale).detach()
            W_fit=nn.Parameter(torch.zeros(2,1,device=device))
            pre=torch.optim.Adam([W_fit],lr=0.05)
            for _ in range(300):
                pred=kappa_scale*torch.tanh(kappa_raw@W_fit).squeeze(-1)
                l=((pred-target)**2).mean()
                pre.zero_grad(); l.backward(); pre.step()
            # copy the fitted values into W with NO graph history
            with torch.no_grad():
                W.data.copy_(W_fit.data)
            if verbose: print(f'  warm-start fit MSE to graded law: {l.item():.4f}')
        fopt=torch.optim.Adam([W],lr=0.2,weight_decay=1e-4)
    kap_frozen=graded_kappa(kappa_scale).detach() if mode=='graded' else None
    def kap():
        if mode=='const': return torch.zeros(N,device=device)
        if mode=='graded': return kap_frozen
        return kappa_scale*torch.tanh(kappa_raw@W).squeeze(-1)
    E=np.array(edges_idx)
    for ep in range(epochs):
        eff=lr*burnin_mult if ep<burnin else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            k=kap()   # recompute inside batch loop -> fresh graph each batch
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            base_p=model.manifold.dist(ea,ep_); base_n=model.manifold.dist(ea.unsqueeze(1),en)
            if mode=='const':
                dp,dn=base_p,base_n
            else:
                dp=base_p*torch.exp(alpha*0.5*(k[a]+k[p]))
                dn=base_n*torch.exp(alpha*0.5*(k[a].unsqueeze(1)+k[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+lambda_h*torch.relu(an-pn+margin).mean()
            opt.zero_grad()
            if fopt: fopt.zero_grad()
            loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
            if fopt: fopt.step()
        if verbose and ep%20==0:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy()); print(f'  ep{ep} MAP {m:.4f} rank {mr:.0f}')
    return evaluate(model.embeddings.detach().cpu().numpy())

# BASELINES (run once, reuse as reference)
print('computing baselines at d=10 (100 epochs)...')
base_const=run('const',epochs=100); print(f'  const  a=0     : MAP {base_const[0]:.4f} rank {base_const[1]:.0f}')
base_graded=run('graded',epochs=100); print(f'  graded a=-0.8  : MAP {base_graded[0]:.4f} rank {base_graded[1]:.0f}')
print('\nuse these as your reference points for every test below')

computing baselines at d=10 (100 epochs)...
  const  a=0     : MAP 0.7210 rank 489
  graded a=-0.8  : MAP 0.7250 rank 463

use these as your reference points for every test below


In [ ]:
# STANDARD 3-METRIC EVAL — use this everywhere from now on. Always MAP, rank, distortion.
from collections import defaultdict, deque

# build distortion pairs ONCE (graph distance via BFS)
_adj=defaultdict(list)
for u,v in edges_idx: _adj[u].append(v); _adj[v].append(u)
_rng=np.random.default_rng(0); DPAIRS=[]
for s in _rng.integers(0,N,size=2000):
    dd={int(s):0}; q=deque([int(s)])
    while q:
        x=q.popleft()
        for y in _adj[x]:
            if y not in dd: dd[y]=dd[x]+1; q.append(y)
    t=int(_rng.integers(0,N))
    if t in dd and t!=s: DPAIRS.append((int(s),t,dd[t]))

def eval3(pos, dist_fn=None):
    """Returns dict with MAP, mean_rank, median_rank, MRR, distortion. Use everywhere."""
    if dist_fn is None:
        def dist_fn(a, allp):
            diff2=np.sum((allp-a)**2,axis=1); nu=1-np.sum(a**2); nv=1-np.sum(allp**2,axis=1)
            return np.arccosh(np.maximum(1+2*diff2/(nu*nv+1e-12),1.0))
    ranks=[]; aps=[]; rrs=[]
    for (u,v) in EVAL:
        d=dist_fn(pos[u],pos); d[u]=np.inf; order=np.argsort(d)
        r=int(np.where(order==v)[0][0])+1; ranks.append(r); rrs.append(1.0/r)
        truth=nbrs[u]; hit=0; precs=[]
        for j,node in enumerate(order):
            if node in truth: hit+=1; precs.append(hit/(j+1))
            if hit==len(truth): break
        if precs: aps.append(np.mean(precs))
    # scaled distortion
    emb_d=np.array([float(dist_fn(pos[u],pos[v:v+1])[0]) for (u,v,dg) in DPAIRS])
    grf_d=np.array([dg for (u,v,dg) in DPAIRS],dtype=float)
    c=np.dot(emb_d,grf_d)/(np.dot(emb_d,emb_d)+1e-12)
    dist=float(np.mean(np.abs(c*emb_d-grf_d)/grf_d))
    return {'MAP':float(np.mean(aps)),'mean_rank':float(np.mean(ranks)),
            'median_rank':float(np.median(ranks)),'MRR':float(np.mean(rrs)),'distortion':dist}

## T1 — alpha sweep on the GRADED law (never optimized!)
You tuned alpha=-0.8 for the BINARY field. The graded law may want a different alpha.

In [5]:
print('alpha sweep on graded law (vs graded baseline MAP %.4f):' % base_graded[0])
for alpha in [-0.4,-0.6,-0.8,-1.0,-1.3,-1.6,-2.0]:
    m,mr=run('graded',alpha=alpha,epochs=100)
    flag='  <-- beats baseline' if m>base_graded[0]+0.003 else ''
    print(f'  alpha={alpha:>5}: MAP {m:.4f} rank {mr:.0f}{flag}')

alpha sweep on graded law (vs graded baseline MAP 0.7250):
  alpha= -0.4: MAP 0.7172 rank 458
  alpha= -0.6: MAP 0.7234 rank 453
  alpha= -0.8: MAP 0.7250 rank 463
  alpha= -1.0: MAP 0.7337 rank 507  <-- beats baseline
  alpha= -1.3: MAP 0.7460 rank 532  <-- beats baseline
  alpha= -1.6: MAP 0.7482 rank 540  <-- beats baseline
  alpha= -2.0: MAP 0.7526 rank 579  <-- beats baseline


## T2 — kappa_scale on graded (how strongly to apply the law)

In [6]:
print('kappa_scale sweep on graded (baseline MAP %.4f):' % base_graded[0])
for ks in [0.5,0.8,1.0,1.5,2.0,3.0]:
    m,mr=run('graded',kappa_scale=ks,epochs=100)
    flag='  <--' if m>base_graded[0]+0.003 else ''
    print(f'  kappa_scale={ks:>4}: MAP {m:.4f} rank {mr:.0f}{flag}')

kappa_scale sweep on graded (baseline MAP 0.7266):
  kappa_scale= 0.5: MAP 0.7183 rank 423
  kappa_scale= 0.8: MAP 0.7215 rank 401
  kappa_scale= 1.0: MAP 0.7266 rank 394
  kappa_scale= 1.5: MAP 0.7432 rank 412  <--
  kappa_scale= 2.0: MAP 0.7515 rank 439  <--
  kappa_scale= 3.0: MAP 0.7577 rank 518  <--


## T3 — epochs & lr (are we under-training?)

In [7]:
print('epochs sweep on graded (baseline 100ep MAP %.4f):' % base_graded[0])
for ep in [100,200,300,400]:
    m,mr=run('graded',epochs=ep)
    print(f'  epochs={ep:>4}: MAP {m:.4f} rank {mr:.0f}')
print('lr sweep on graded:')
for lr in [5,10,20,50]:
    m,mr=run('graded',lr=lr,epochs=100)
    print(f'  lr={lr:>4}: MAP {m:.4f} rank {mr:.0f}')

epochs sweep on graded (baseline 100ep MAP 0.7266):
  epochs= 100: MAP 0.7266 rank 394
  epochs= 200: MAP 0.7616 rank 371
  epochs= 300: MAP 0.7699 rank 330
  epochs= 400: MAP 0.7711 rank 304
lr sweep on graded:
  lr=   5: MAP 0.7158 rank 404
  lr=  10: MAP 0.7266 rank 394
  lr=  20: MAP 0.7582 rank 433
  lr=  50: MAP 0.7904 rank 359


## T4 — negatives count

In [8]:
print('negatives sweep on graded (baseline K=50 MAP %.4f):' % base_graded[0])
for K in [20,50,100,200]:
    m,mr=run('graded',K=K,epochs=100)
    flag='  <--' if m>base_graded[0]+0.003 else ''
    print(f'  K={K:>4}: MAP {m:.4f} rank {mr:.0f}{flag}')

negatives sweep on graded (baseline K=50 MAP 0.7266):
  K=  20: MAP 0.7302 rank 366  <--
  K=  50: MAP 0.7266 rank 394
  K= 100: MAP 0.7197 rank 448
  K= 200: MAP 0.7238 rank 475


## T5 — WARM-START the learnable field at graded (the real shot)
Start the field near the graded law, then LET IT LEARN. Might beat forced-graded by adapting beyond the pure law.

In [12]:
print('warm-started learnable field vs graded baseline MAP %.4f:' % base_graded[0])
for seed in [0,1,2]:
    m,mr=run('learned',warmstart_graded=True,epochs=150,seed=seed)
    flag='  <-- beats graded!' if m>base_graded[0]+0.003 else ''
    print(f'  warmstart seed{seed}: MAP {m:.4f} rank {mr:.0f}{flag}')
print('(compare to cold-start learned, which collapsed to binary ~0.750)')

warm-started learnable field vs graded baseline MAP 0.7250:
  warmstart seed0: MAP 0.7024 rank 172
  warmstart seed1: MAP 0.7013 rank 184
  warmstart seed2: MAP 0.7094 rank 223
(compare to cold-start learned, which collapsed to binary ~0.750)


## T6 — multi-seed confirmation (run this on ANY promising config)

In [13]:
def multi_seed(label, **kw):
    ms=[]; rs=[]
    for s in [0,1,2]:
        m,mr=run(seed=s,**kw); ms.append(m); rs.append(mr)
    print(f'{label}: MAP {np.mean(ms):.4f} +/- {np.std(ms):.4f}   rank {np.mean(rs):.0f} +/- {np.std(rs):.0f}')
    return np.mean(ms),np.std(ms)

# example: confirm graded is really better than const
multi_seed('const  ', mode='const', epochs=100)
multi_seed('graded ', mode='graded', alpha=-0.8, epochs=100)
# add your promising config from T1-T5 here:
# multi_seed('BEST', mode='graded', alpha=..., kappa_scale=..., epochs=...)

const  : MAP 0.7237 +/- 0.0019   rank 546 +/- 48
graded : MAP 0.7248 +/- 0.0021   rank 504 +/- 39


(np.float64(0.7247827841250981), np.float64(0.0020838252879814253))

## How to use this sandbox
1. Run cells 1-4 once (sets up baselines).
2. Run any test T1-T5 — each is independent, ~1-3 min.
3. When a test shows a config beating baseline by >0.003, confirm it in T6 with multi-seed.
4. Only trust gains that survive multi-seed (std ~0.0015, so need >0.003 to be real).

**Most promising bets:** T1 (alpha never tuned for graded), T5 (warm-start might unlock a learnable graded field). T3 will tell you if you're just under-training.

In [14]:
# ============================================================
# CONVERGENCE + LR STABILITY TEST
# Trains long, prints the MAP curve every 25 epochs, so you can SEE where it
# plateaus. Runs a few lr values x 2 seeds to check the lr=50 result is real
# and stable, not a lucky high-lr bounce.
# ============================================================
def run_verbose_curve(mode='graded', dim=10, alpha=-0.8, lr=10.0, epochs=400,
                      K=50, seed=0, check_every=25):
    """Same as run() but reports MAP at intervals so we see the convergence curve."""
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kap_frozen=graded_kappa(1.0).detach()
    def kap():
        if mode=='const': return torch.zeros(N,device=device)
        return kap_frozen
    E=np.array(edges_idx); curve=[]
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            k=kap()
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            if mode=='const':
                dp,dn=bp,bn
            else:
                dp=bp*torch.exp(alpha*0.5*(k[a]+k[p])); dn=bn*torch.exp(alpha*0.5*(k[a].unsqueeze(1)+k[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%check_every==0 or ep==epochs-1:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            curve.append((ep+1,m,mr))
            print(f'    ep{ep+1:>3}: MAP {m:.4f}  rank {mr:.0f}')
    return curve

# ---- lr stability x convergence: graded law, d=10 ----
for lr in [20, 50]:
    print(f'\n===== lr={lr}, graded, d=10 (watch for plateau + seed agreement) =====')
    finals=[]
    for seed in [0,1]:
        print(f'  seed {seed}:')
        curve=run_verbose_curve(mode='graded', lr=lr, epochs=400, seed=seed)
        finals.append(curve[-1][1])
    spread=abs(finals[0]-finals[1])
    print(f'  >> lr={lr} final MAP: seed0 {finals[0]:.4f}, seed1 {finals[1]:.4f}, '
          f'spread {spread:.4f}  -> {"STABLE" if spread<0.006 else "UNSTABLE (high variance)"}')


===== lr=20, graded, d=10 (watch for plateau + seed agreement) =====
  seed 0:
    ep 25: MAP 0.7199  rank 493
    ep 50: MAP 0.7270  rank 490
    ep 75: MAP 0.7417  rank 501
    ep100: MAP 0.7542  rank 501
    ep125: MAP 0.7607  rank 472
    ep150: MAP 0.7659  rank 442
    ep175: MAP 0.7667  rank 414
    ep200: MAP 0.7693  rank 395
    ep225: MAP 0.7720  rank 372
    ep250: MAP 0.7740  rank 356
    ep275: MAP 0.7766  rank 342
    ep300: MAP 0.7786  rank 330
    ep325: MAP 0.7796  rank 317
    ep350: MAP 0.7803  rank 308
    ep375: MAP 0.7818  rank 302
    ep400: MAP 0.7831  rank 290
  seed 1:
    ep 25: MAP 0.7260  rank 690
    ep 50: MAP 0.7282  rank 671
    ep 75: MAP 0.7423  rank 661
    ep100: MAP 0.7541  rank 661
    ep125: MAP 0.7620  rank 626
    ep150: MAP 0.7669  rank 595
    ep175: MAP 0.7677  rank 565
    ep200: MAP 0.7716  rank 530
    ep225: MAP 0.7734  rank 502
    ep250: MAP 0.7746  rank 476
    ep275: MAP 0.7778  rank 452
    ep300: MAP 0.7782  rank 432
    ep325: MAP

In [17]:
CONV_LR = 50      # T7 confirmed this is STABLE
CONV_EP = 400     # still climbing, but fine for a fair comparison

print(f'Converged fair comparison: lr={CONV_LR}, epochs={CONV_EP}, 3 seeds\n')
results={}
for mode in ['const','graded','learned']:
    ms=[]; rs=[]
    for seed in [0,1,2]:
        m,mr=run(mode=mode, alpha=-0.8, lr=CONV_LR, epochs=CONV_EP, seed=seed)
        ms.append(m); rs.append(mr)
    results[mode]=(np.mean(ms),np.std(ms))
    print(f'  {mode:<8}: MAP {np.mean(ms):.4f} +/- {np.std(ms):.4f}   rank {np.mean(rs):.0f} +/- {np.std(rs):.0f}')

cm=results['const'][0]; gm=results['graded'][0]; lm=results['learned'][0]
print('\n--- verdicts (need gap > 0.004 to beat noise) ---')
print(f'  graded vs const:   {gm-cm:+.4f}  -> {"graded WINS" if gm>cm+0.004 else "const WINS" if cm>gm+0.004 else "TIE"}')
print(f'  graded vs learned: {gm-lm:+.4f}  -> {"graded WINS" if gm>lm+0.004 else "learned WINS" if lm>gm+0.004 else "TIE"}')
print(f'  learned vs const:  {lm-cm:+.4f}  -> {"learned WINS" if lm>cm+0.004 else "const WINS" if cm>lm+0.004 else "TIE"}')

Converged fair comparison: lr=50, epochs=400, 3 seeds

  const   : MAP 0.7783 +/- 0.0050   rank 135 +/- 16
  graded  : MAP 0.8172 +/- 0.0015   rank 227 +/- 22
  learned : MAP 0.7354 +/- 0.0040   rank 21 +/- 2

--- verdicts (need gap > 0.004 to beat noise) ---
  graded vs const:   +0.0389  -> graded WINS
  graded vs learned: +0.0818  -> graded WINS
  learned vs const:  -0.0429  -> const WINS


In [18]:
# Euclidean at convergence — its OWN recipe (Adam, not RiemannianSGD).
# lr=50 is a hyperbolic/Riemannian setting; Euclidean uses Adam at its own lr.
# Trained long to match the "converged" standard of the other three.
def run_euclid(dim=10, epochs=400, K=50, lr=0.1, init_scale=1e-3, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    emb=nn.Parameter(torch.empty(N,dim,device=device).uniform_(-init_scale,init_scale))
    opt=torch.optim.Adam([emb],lr=lr)
    E=np.array(edges_idx)
    for ep in range(epochs):
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=emb[a]; ep_=emb[p]; en=emb[ng]
            dp=((ea-ep_)**2).sum(-1); dn=((ea.unsqueeze(1)-en)**2).sum(-1)
            loss=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            opt.zero_grad(); loss.backward(); opt.step()
    pos=emb.detach().cpu().numpy()
    # eval with EUCLIDEAN distance (not poincare!)
    def d_euclid(a,allp): return np.linalg.norm(allp-a,axis=1)
    ranks=[]; aps=[]
    for (u,v) in EVAL:
        d=d_euclid(pos[u],pos); d[u]=np.inf; order=np.argsort(d)
        ranks.append(int(np.where(order==v)[0][0])+1)
        truth=nbrs[u]; hit=0; precs=[]
        for j,node in enumerate(order):
            if node in truth: hit+=1; precs.append(hit/(j+1))
            if hit==len(truth): break
        if precs: aps.append(np.mean(precs))
    return float(np.mean(aps)), float(np.mean(ranks))

print('Euclidean at d=10, 400 epochs, 3 seeds:')
ms=[]; rs=[]
for seed in [0,1,2]:
    m,mr=run_euclid(dim=10, epochs=400, seed=seed); ms.append(m); rs.append(mr)
    print(f'  seed{seed}: MAP {m:.4f}  rank {mr:.0f}')
print(f'  Euclidean: MAP {np.mean(ms):.4f} +/- {np.std(ms):.4f}   rank {np.mean(rs):.0f} +/- {np.std(rs):.0f}')

Euclidean at d=10, 400 epochs, 3 seeds:
  seed0: MAP 0.6541  rank 15
  seed1: MAP 0.6711  rank 13
  seed2: MAP 0.6522  rank 13
  Euclidean: MAP 0.6591 +/- 0.0085   rank 13 +/- 1


In [20]:
# Converged-to-plateau: all FOUR methods to 800 epochs, curve printed
def d_euclid_np(a,allp): return np.linalg.norm(allp-a,axis=1)

def evaluate_euclid(pos):
    ranks=[]; aps=[]
    for (u,v) in EVAL:
        d=d_euclid_np(pos[u],pos); d[u]=np.inf; order=np.argsort(d)
        ranks.append(int(np.where(order==v)[0][0])+1)
        truth=nbrs[u]; hit=0; precs=[]
        for j,node in enumerate(order):
            if node in truth: hit+=1; precs.append(hit/(j+1))
            if hit==len(truth): break
        if precs: aps.append(np.mean(precs))
    return float(np.mean(aps)), float(np.mean(ranks))

def run_long(mode, epochs=800, lr=50, alpha=-0.8, K=50, seed=0, check_every=100):
    torch.manual_seed(seed); np.random.seed(seed)
    if mode=='euclid':
        # Euclidean: plain params, Adam, its own lr, Euclidean eval
        emb=nn.Parameter(torch.empty(N,10,device=device).uniform_(-1e-3,1e-3))
        opt=torch.optim.Adam([emb],lr=0.1)
        E=np.array(edges_idx)
        for ep in range(epochs):
            perm=np.random.permutation(len(E))
            for bs in range(0,len(E),1024):
                b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
                ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
                ea=emb[a]; ep_=emb[p]; en=emb[ng]
                dp=((ea-ep_)**2).sum(-1); dn=((ea.unsqueeze(1)-en)**2).sum(-1)
                loss=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
                opt.zero_grad(); loss.backward(); opt.step()
            if (ep+1)%check_every==0 or ep==epochs-1:
                m,mr=evaluate_euclid(emb.detach().cpu().numpy())
                print(f'    {mode:<8} ep{ep+1:>3}: MAP {m:.4f} rank {mr:.0f}')
        return evaluate_euclid(emb.detach().cpu().numpy())

    # hyperbolic methods: const / graded / learned
    model=PoincareEmbedding(N,10,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    W=None; fopt=None
    if mode=='learned':
        W=nn.Parameter(torch.zeros(2,1,device=device)); fopt=torch.optim.Adam([W],lr=0.2,weight_decay=1e-4)
    def kap():
        if mode=='const': return torch.zeros(N,device=device)
        if mode=='graded': return kf
        return torch.tanh(kappa_raw@W).squeeze(-1)
    E=np.array(edges_idx)
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            k=kap(); b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            if mode=='const': dp,dn=bp,bn
            else:
                dp=bp*torch.exp(alpha*0.5*(k[a]+k[p])); dn=bn*torch.exp(alpha*0.5*(k[a].unsqueeze(1)+k[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad()
            if fopt: fopt.zero_grad()
            loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
            if fopt: fopt.step()
        if (ep+1)%check_every==0 or ep==epochs-1:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            print(f'    {mode:<8} ep{ep+1:>3}: MAP {m:.4f} rank {mr:.0f}')
    return evaluate(model.embeddings.detach().cpu().numpy())

finals={}
for mode in ['euclid','const','graded','learned']:
    print(f'\n=== {mode} to 800ep (seed 0) ===')
    finals[mode]=run_long(mode, epochs=800, seed=0)

print('\n=== 800-epoch summary (seed 0) ===')
for mode in ['euclid','const','graded','learned']:
    m,mr=finals[mode]; print(f'  {mode:<8}: MAP {m:.4f}  rank {mr:.0f}')


=== euclid to 800ep (seed 0) ===
    euclid   ep100: MAP 0.5287 rank 34
    euclid   ep200: MAP 0.6078 rank 17
    euclid   ep300: MAP 0.6352 rank 16
    euclid   ep400: MAP 0.6541 rank 15
    euclid   ep500: MAP 0.6801 rank 12
    euclid   ep600: MAP 0.6955 rank 10
    euclid   ep700: MAP 0.7113 rank 9
    euclid   ep800: MAP 0.7170 rank 8

=== const to 800ep (seed 0) ===
    const    ep100: MAP 0.7522 rank 426
    const    ep200: MAP 0.7711 rank 188
    const    ep300: MAP 0.7724 rank 140
    const    ep400: MAP 0.7716 rank 127
    const    ep500: MAP 0.7728 rank 119
    const    ep600: MAP 0.7736 rank 113
    const    ep700: MAP 0.7735 rank 113
    const    ep800: MAP 0.7716 rank 109

=== graded to 800ep (seed 0) ===
    graded   ep100: MAP 0.7909 rank 366
    graded   ep200: MAP 0.8063 rank 275
    graded   ep300: MAP 0.8147 rank 248
    graded   ep400: MAP 0.8189 rank 219
    graded   ep500: MAP 0.8268 rank 206
    graded   ep600: MAP 0.8278 rank 206
    graded   ep700: MAP 0.830

In [5]:
# Graded to 2000 epochs — find the true MAP ceiling (the only method still climbing)
def run_graded_long(epochs=2000, lr=50, alpha=-0.8, K=50, seed=0, check_every=200):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,10,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    E=np.array(edges_idx); best=0; best_ep=0
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            dp=bp*torch.exp(alpha*0.5*(kf[a]+kf[p])); dn=bn*torch.exp(alpha*0.5*(kf[a].unsqueeze(1)+kf[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%check_every==0 or ep==epochs-1:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            flag=''
            if m>best: best=m; best_ep=ep+1; flag='  <-- best'
            print(f'    ep{ep+1:>4}: MAP {m:.4f} rank {mr:.0f}{flag}')
    print(f'\n  BEST: MAP {best:.4f} at ep{best_ep}')
    return best

run_graded_long(epochs=2000, seed=0)

    ep 200: MAP 0.8063 rank 275  <-- best
    ep 400: MAP 0.8189 rank 219  <-- best
    ep 600: MAP 0.8278 rank 206  <-- best
    ep 800: MAP 0.8317 rank 192  <-- best
    ep1000: MAP 0.8344 rank 191  <-- best
    ep1200: MAP 0.8380 rank 191  <-- best
    ep1400: MAP 0.8398 rank 187  <-- best
    ep1600: MAP 0.8408 rank 187  <-- best
    ep1800: MAP 0.8417 rank 188  <-- best
    ep2000: MAP 0.8428 rank 182  <-- best

  BEST: MAP 0.8428 at ep2000


0.8427510875257898

In [11]:
# Converged alpha sweep: which alpha is best AT CONVERGENCE (not at 100 epochs)?
def run_graded_alpha(alpha, epochs=1500, lr=50, K=50, seed=0, check_every=300):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,10,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    E=np.array(edges_idx); best=0
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            dp=bp*torch.exp(alpha*0.5*(kf[a]+kf[p])); dn=bn*torch.exp(alpha*0.5*(kf[a].unsqueeze(1)+kf[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%check_every==0:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            if m>best: best=m
            print(f'    a={alpha} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
    return best

print('CONVERGED alpha sweep (each to 1500 epochs):')
for alpha in [-1.3, -2.0, -2.5]:
    b=run_graded_alpha(alpha, epochs=1500)
    print(f'  >> alpha={alpha}: converged MAP {b:.4f}\n')

CONVERGED alpha sweep (each to 1500 epochs):
    a=-1.3 ep300: MAP 0.8311 rank 270
    a=-1.3 ep600: MAP 0.8504 rank 249
    a=-1.3 ep900: MAP 0.8583 rank 225
    a=-1.3 ep1200: MAP 0.8600 rank 228
    a=-1.3 ep1500: MAP 0.8640 rank 220
  >> alpha=-1.3: converged MAP 0.8640

    a=-2.0 ep300: MAP 0.8483 rank 328
    a=-2.0 ep600: MAP 0.8654 rank 309
    a=-2.0 ep900: MAP 0.8734 rank 272
    a=-2.0 ep1200: MAP 0.8785 rank 273
    a=-2.0 ep1500: MAP 0.8810 rank 263
  >> alpha=-2.0: converged MAP 0.8810

    a=-2.5 ep300: MAP 0.8435 rank 382
    a=-2.5 ep600: MAP 0.8642 rank 356
    a=-2.5 ep900: MAP 0.8761 rank 309
    a=-2.5 ep1200: MAP 0.8788 rank 299
    a=-2.5 ep1500: MAP 0.8831 rank 287
  >> alpha=-2.5: converged MAP 0.8831



In [13]:
# ============================================================
# HEADLINE NUMBER: graded @ alpha=-2.0, multi-seed, to convergence
# Locks the official result with error bars. 3 seeds, 2000 epochs.
# ============================================================
def run_graded_final(alpha=-2.0, epochs=2000, lr=50, K=50, seed=0, check_every=500):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,10,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    E=np.array(edges_idx); best_m=0; best_r=0
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            dp=bp*torch.exp(alpha*0.5*(kf[a]+kf[p])); dn=bn*torch.exp(alpha*0.5*(kf[a].unsqueeze(1)+kf[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%check_every==0:
            pos=model.embeddings.detach().cpu().numpy()
            assert not np.isnan(pos).any(), f"NaN at ep{ep+1}!"   # stability guard
            m,mr=evaluate(pos)
            if m>best_m: best_m=m; best_r=mr
            print(f'    seed{seed} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
    return best_m, best_r

print('HEADLINE: graded @ alpha=-2.0, 3 seeds, 2000 epochs\n')
maps=[]; ranks=[]
for seed in [1,2]:
    m,mr=run_graded_final(alpha=-2.0, epochs=2000, seed=seed)
    maps.append(m); ranks.append(mr)
    print(f'  >> seed{seed}: MAP {m:.4f}  rank {mr:.0f}\n')

print('='*50)
print(f'HEADLINE NUMBER: MAP {np.mean(maps):.4f} +/- {np.std(maps):.4f}   rank {np.mean(ranks):.0f} +/- {np.std(ranks):.0f}')
print(f'  vs constant-curvature baseline 0.772  ->  +{np.mean(maps)-0.772:.4f} MAP')
print('='*50)

HEADLINE: graded @ alpha=-2.0, 3 seeds, 2000 epochs

    seed1 ep500: MAP 0.8609 rank 343
    seed1 ep1000: MAP 0.8744 rank 317
    seed1 ep1500: MAP 0.8788 rank 288
    seed1 ep2000: MAP 0.8830 rank 280
  >> seed1: MAP 0.8830  rank 280

    seed2 ep500: MAP 0.8585 rank 344
    seed2 ep1000: MAP 0.8718 rank 309
    seed2 ep1500: MAP 0.8782 rank 299
    seed2 ep2000: MAP 0.8815 rank 277
  >> seed2: MAP 0.8815  rank 277

HEADLINE NUMBER: MAP 0.8823 +/- 0.0007   rank 278 +/- 1
  vs constant-curvature baseline 0.772  ->  +0.1103 MAP


In [18]:
# Find graded's true plateau: run long, stop reporting when gain flattens
def run_to_plateau(alpha=-2.0, max_epochs=5000, lr=50, K=50, seed=0, check_every=500, flat_thresh=0.001):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,10,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    E=np.array(edges_idx); prev=0
    for ep in range(max_epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            dp=bp*torch.exp(alpha*0.5*(kf[a]+kf[p])); dn=bn*torch.exp(alpha*0.5*(kf[a].unsqueeze(1)+kf[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%check_every==0:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            gain=m-prev
            print(f'    ep{ep+1}: MAP {m:.4f} rank {mr:.0f}  (gain +{gain:.4f})')
            if 0<gain<flat_thresh:
                print(f'\n  PLATEAU reached at ep{ep+1}: MAP {m:.4f}')
                return ep+1, m
            prev=m
    print(f'\n  hit max_epochs, still improving slightly: MAP {m:.4f}')
    return max_epochs, m

plateau_ep, plateau_map = run_to_plateau(alpha=-2.0)
print(f'\n>>> Use {plateau_ep} epochs as the budget for all methods. Graded peak: {plateau_map:.4f}')

    ep500: MAP 0.8640 rank 301  (gain +0.8640)
    ep1000: MAP 0.8734 rank 274  (gain +0.0094)
    ep1500: MAP 0.8810 rank 263  (gain +0.0076)
    ep2000: MAP 0.8858 rank 243  (gain +0.0048)
    ep2500: MAP 0.8888 rank 235  (gain +0.0030)
    ep3000: MAP 0.8903 rank 227  (gain +0.0014)
    ep3500: MAP 0.8920 rank 220  (gain +0.0017)
    ep4000: MAP 0.8930 rank 221  (gain +0.0010)
    ep4500: MAP 0.8939 rank 206  (gain +0.0009)

  PLATEAU reached at ep4500: MAP 0.8939

>>> Use 4500 epochs as the budget for all methods. Graded peak: 0.8939


In [19]:
# ============================================================
# FINAL FAIR COMPARISON — all methods at graded's plateau budget (3500 ep)
# graded & const: same budget (both converge). euclid: same budget (still gains).
# learned: report PEAK, not ep3500 (it degrades) — tracked separately.
# 2 seeds each for error bars (bump to 3 if you have time).
# ============================================================
BUDGET = 4500

def train_final(mode, alpha=-2.0, epochs=BUDGET, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    if mode=='euclid':
        emb=nn.Parameter(torch.empty(N,10,device=device).uniform_(-1e-3,1e-3))
        opt=torch.optim.Adam([emb],lr=0.1); E=np.array(edges_idx)
        for ep in range(epochs):
            perm=np.random.permutation(len(E))
            for bs in range(0,len(E),1024):
                b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
                ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
                ea=emb[a]; ep_=emb[p]; en=emb[ng]
                dp=((ea-ep_)**2).sum(-1); dn=((ea.unsqueeze(1)-en)**2).sum(-1)
                loss=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
                opt.zero_grad(); loss.backward(); opt.step()
        pos=emb.detach().cpu().numpy()
        def de(a,allp): return np.linalg.norm(allp-a,axis=1)
        ranks=[]; aps=[]
        for (u,v) in EVAL:
            d=de(pos[u],pos); d[u]=np.inf; order=np.argsort(d)
            ranks.append(int(np.where(order==v)[0][0])+1)
            truth=nbrs[u]; hit=0; precs=[]
            for j,node in enumerate(order):
                if node in truth: hit+=1; precs.append(hit/(j+1))
                if hit==len(truth): break
            if precs: aps.append(np.mean(precs))
        return float(np.mean(aps)), float(np.mean(ranks))

    model=PoincareEmbedding(N,10,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    W=None; fopt=None
    if mode=='learned':
        W=nn.Parameter(torch.zeros(2,1,device=device)); fopt=torch.optim.Adam([W],lr=0.2,weight_decay=1e-4)
    def kap():
        if mode=='const': return torch.zeros(N,device=device)
        if mode=='graded': return kf
        return torch.tanh(kappa_raw@W).squeeze(-1)
    E=np.array(edges_idx); best_m=0; best_r=0
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            k=kap(); b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            if mode=='const': dp,dn=bp,bn
            else:
                dp=bp*torch.exp(alpha*0.5*(k[a]+k[p])); dn=bn*torch.exp(alpha*0.5*(k[a].unsqueeze(1)+k[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad()
            if fopt: fopt.zero_grad()
            loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
            if fopt: fopt.step()
        # track PEAK for learned (it degrades); for others peak==final anyway
        if mode=='learned' and (ep+1)%250==0:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            if m>best_m: best_m=m; best_r=mr
    pos=model.embeddings.detach().cpu().numpy()
    m,mr=evaluate(pos)
    if mode=='learned':
        if m>best_m: best_m=m; best_r=mr
        return best_m, best_r   # report learned's PEAK
    return m, mr

print(f'FINAL COMPARISON at {BUDGET} epochs, 2 seeds each\n')
final={}
for mode in ['graded','const','euclid','learned']:
    ms=[]; rs=[]
    for seed in [0,1]:
        m,mr=train_final(mode, seed=seed); ms.append(m); rs.append(mr)
    final[mode]=(np.mean(ms),np.std(ms),np.mean(rs),np.std(rs))
    tag=' (PEAK, degrades after)' if mode=='learned' else ''
    print(f'  {mode:<8}: MAP {np.mean(ms):.4f} +/- {np.std(ms):.4f}   rank {np.mean(rs):.0f}{tag}')

print('\n'+'='*55)
g=final['graded'][0]; c=final['const'][0]
print(f'HEADLINE: graded {g:.4f} vs const {c:.4f}  ->  +{g-c:.4f} MAP')
print('='*55)

FINAL COMPARISON at 4500 epochs, 2 seeds each

  graded  : MAP 0.8934 +/- 0.0004   rank 210
  const   : MAP 0.7753 +/- 0.0047   rank 118
  euclid  : MAP 0.8671 +/- 0.0013   rank 5
  learned : MAP 0.3326 +/- 0.0048   rank 1551 (PEAK, degrades after)

HEADLINE: graded 0.8934 vs const 0.7753  ->  +0.1182 MAP


In [6]:
import os
CKPT_DIR = HME_ROOT + '/results/dim_runs'
os.makedirs(CKPT_DIR, exist_ok=True)

def train_dim_logged(mode, dim, alpha=-2.0, epochs=4500, lr=50, K=50, seed=0, log_every=100):
    torch.manual_seed(seed); np.random.seed(seed)
    ckpt=f'{CKPT_DIR}/{mode}_d{dim}.pt'
    curve=[]

    if mode=='euclid':
        emb=nn.Parameter(torch.empty(N,dim,device=device).uniform_(-1e-3,1e-3))
        opt=torch.optim.Adam([emb],lr=0.1)
        start=0
        if os.path.exists(ckpt):
            ck=torch.load(ckpt,map_location=device); emb.data=ck['emb'].to(device)
            opt.load_state_dict(ck['opt']); start=ck['ep']; curve=ck['curve']
            print(f'  resumed {mode} d{dim} from ep{start}')
        E=np.array(edges_idx)
        def de(a,allp): return np.linalg.norm(allp-a,axis=1)
        for ep in range(start,epochs):
            perm=np.random.permutation(len(E))
            for bs in range(0,len(E),1024):
                b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
                ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
                ea=emb[a]; ep_=emb[p]; en=emb[ng]
                dp=((ea-ep_)**2).sum(-1); dn=((ea.unsqueeze(1)-en)**2).sum(-1)
                loss=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
                opt.zero_grad(); loss.backward(); opt.step()
            if (ep+1)%log_every==0:
                pos=emb.detach().cpu().numpy(); ranks=[]; aps=[]
                for (u,v) in EVAL:
                    d=de(pos[u],pos); d[u]=np.inf; order=np.argsort(d)
                    ranks.append(int(np.where(order==v)[0][0])+1)
                    truth=nbrs[u]; hit=0; precs=[]
                    for j,node in enumerate(order):
                        if node in truth: hit+=1; precs.append(hit/(j+1))
                        if hit==len(truth): break
                    if precs: aps.append(np.mean(precs))
                m,mr=float(np.mean(aps)),float(np.mean(ranks)); curve.append((ep+1,m,mr))
                print(f'    {mode} d{dim} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
                torch.save({'emb':emb.detach().cpu(),'opt':opt.state_dict(),'ep':ep+1,'curve':curve}, ckpt)
        return curve

    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    def kap(): return torch.zeros(N,device=device) if mode=='const' else kf
    start=0
    if os.path.exists(ckpt):
        ck=torch.load(ckpt,map_location=device); model.load_state_dict(ck['model'])
        opt.load_state_dict(ck['opt']); start=ck['ep']; curve=ck['curve']
        print(f'  resumed {mode} d{dim} from ep{start}')
    E=np.array(edges_idx)
    for ep in range(start,epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            k=kap(); b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            if mode=='const': dp,dn=bp,bn
            else:
                dp=bp*torch.exp(alpha*0.5*(k[a]+k[p])); dn=bn*torch.exp(alpha*0.5*(k[a].unsqueeze(1)+k[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%log_every==0:
            pos=model.embeddings.detach().cpu().numpy()
            if np.isnan(pos).any():
                print(f'    !! {mode} d{dim} NaN at ep{ep+1} - alpha={alpha} too strong for this dim'); break
            m,mr=evaluate(pos); curve.append((ep+1,m,mr))
            print(f'    {mode} d{dim} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
            torch.save({'model':model.state_dict(),'opt':opt.state_dict(),'ep':ep+1,'curve':curve}, ckpt)
    return curve

# run d=5 and d=2, all three methods, resumable
all_curves={}
for dim in [10, 5,2]:
    for mode in ['graded','const','euclid']:
        print(f'\n=== {mode} d={dim} ===')
        all_curves[(mode,dim)]=train_dim_logged(mode, dim, epochs=3000)


=== graded d=10 ===
    graded d10 ep100: MAP 0.8121 rank 449
    graded d10 ep200: MAP 0.8346 rank 370
    graded d10 ep300: MAP 0.8483 rank 328
    graded d10 ep400: MAP 0.8572 rank 313
    graded d10 ep500: MAP 0.8640 rank 301
    graded d10 ep600: MAP 0.8654 rank 309
    graded d10 ep700: MAP 0.8692 rank 291
    graded d10 ep800: MAP 0.8690 rank 280
    graded d10 ep900: MAP 0.8734 rank 272
    graded d10 ep1000: MAP 0.8734 rank 274
    graded d10 ep1100: MAP 0.8784 rank 270
    graded d10 ep1200: MAP 0.8785 rank 273
    graded d10 ep1300: MAP 0.8802 rank 269
    graded d10 ep1400: MAP 0.8808 rank 262
    graded d10 ep1500: MAP 0.8810 rank 263
    graded d10 ep1600: MAP 0.8857 rank 257
    graded d10 ep1700: MAP 0.8839 rank 249
    graded d10 ep1800: MAP 0.8841 rank 259
    graded d10 ep1900: MAP 0.8854 rank 253
    graded d10 ep2000: MAP 0.8858 rank 243
    graded d10 ep2100: MAP 0.8858 rank 246
    graded d10 ep2200: MAP 0.8873 rank 245
    graded d10 ep2300: MAP 0.8876 rank 243

In [8]:
import os, torch, numpy as np
CKPT_DIR = HME_ROOT + '/results/dim_runs'

# distortion pairs: graph (BFS) distance vs embedded distance, for a sample of node pairs
from collections import defaultdict, deque
adj=defaultdict(list)
for u,v in edges_idx: adj[u].append(v); adj[v].append(u)
rng=np.random.default_rng(0); DPAIRS=[]
for s in rng.integers(0,N,size=2000):
    dd={int(s):0}; q=deque([int(s)])
    while q:
        x=q.popleft()
        for y in adj[x]:
            if y not in dd: dd[y]=dd[x]+1; q.append(y)
    t=int(rng.integers(0,N))
    if t in dd and t!=s: DPAIRS.append((int(s),t,dd[t]))

def d_poincare(a,allp):
    diff2=np.sum((allp-a)**2,axis=1); nu=1-np.sum(a**2); nv=1-np.sum(allp**2,axis=1)
    return np.arccosh(np.maximum(1+2*diff2/(nu*nv+1e-12),1.0))
def d_euclid(a,allp): return np.linalg.norm(allp-a,axis=1)

# SCALED distortion (De Sa comparable): fit best global scale, then mean relative error
def distortion_scaled(pos, dist_fn):
    emb_d=np.array([float(dist_fn(pos[u], pos[v:v+1])[0]) for (u,v,dg) in DPAIRS])
    grf_d=np.array([dg for (u,v,dg) in DPAIRS], dtype=float)
    c=np.dot(emb_d,grf_d)/(np.dot(emb_d,emb_d)+1e-12)
    return float(np.mean(np.abs(c*emb_d-grf_d)/grf_d))

print(f"{'method':<8}{'dim':>4}{'distortion':>12}")
for dim in [10,5,2]:
    for mode in ['graded','const','euclid']:
        f=f'{CKPT_DIR}/{mode}_d{dim}.pt'
        if not os.path.exists(f):
            print(f'{mode:<8}{dim:>4}   (no checkpoint yet)'); continue
        ck=torch.load(f, map_location='cpu')
        if mode=='euclid':
            pos=ck['emb'].numpy(); dfn=d_euclid
        else:
            sd=ck['model']; key=[k for k in sd if 'embed' in k.lower()][0]
            pos=sd[key].numpy(); dfn=d_poincare
        if np.isnan(pos).any():
            print(f'{mode:<8}{dim:>4}   (NaN embedding)'); continue
        print(f'{mode:<8}{dim:>4}{distortion_scaled(pos,dfn):>12.4f}')

method   dim  distortion
graded    10      0.1460
const     10      0.1109
euclid    10      0.1935
graded     5      0.1520
const      5      0.1161
euclid     5      0.2115
graded     2      0.2180
const      2      0.1639
euclid     2      0.3745


In [8]:
import os
CKPT_DIR = HME_ROOT + '/results/dim_runs'
for dim in [10, 5, 2]:
    print(f'\n=== learned d={dim} (saving peak) ===')
    torch.manual_seed(0); np.random.seed(0)
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=50)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    W=nn.Parameter(torch.zeros(2,1,device=device)); fopt=torch.optim.Adam([W],lr=0.2,weight_decay=1e-4)
    E=np.array(edges_idx); best=0; best_ep=0
    for ep in range(2000):
        eff=50*0.01 if ep<10 else 50
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            k=torch.tanh(kappa_raw@W).squeeze(-1)
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],50)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            dp=bp*torch.exp(-0.8*0.5*(k[a]+k[p])); dn=bn*torch.exp(-0.8*0.5*(k[a].unsqueeze(1)+k[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); fopt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step(); fopt.step()
        if (ep+1)%100==0:
            pos=model.embeddings.detach().cpu().numpy()
            m,mr=evaluate(pos)
            if m>best:
                best=m; best_ep=ep+1
                torch.save({'emb':torch.tensor(pos),'ep':best_ep,'best_map':best},
                           f'{CKPT_DIR}/learned_d{dim}.pt')   # SAVE peak to Drive now
                print(f'    ep{ep+1}: MAP {m:.4f} rank {mr:.0f}  <-- peak SAVED')
            else:
                print(f'    ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
    print(f'  learned d={dim} PEAK: MAP {best:.4f} saved')


=== learned d=10 (saving peak) ===
    ep100: MAP 0.7577 rank 41  <-- peak SAVED
    ep200: MAP 0.7563 rank 25
    ep300: MAP 0.7511 rank 21
    ep400: MAP 0.7397 rank 18
    ep500: MAP 0.7223 rank 17
    ep600: MAP 0.7045 rank 17
    ep700: MAP 0.6854 rank 16
    ep800: MAP 0.6691 rank 16
    ep900: MAP 0.6578 rank 16
    ep1000: MAP 0.6520 rank 16
    ep1100: MAP 0.6471 rank 16
    ep1200: MAP 0.6444 rank 17
    ep1300: MAP 0.6435 rank 16
    ep1400: MAP 0.6451 rank 17
    ep1500: MAP 0.6445 rank 17
    ep1600: MAP 0.6468 rank 16
    ep1700: MAP 0.6453 rank 17
    ep1800: MAP 0.6433 rank 17
    ep1900: MAP 0.6443 rank 17
    ep2000: MAP 0.6443 rank 16
  learned d=10 PEAK: MAP 0.7577 saved

=== learned d=5 (saving peak) ===
    ep100: MAP 0.6723 rank 188  <-- peak SAVED
    ep200: MAP 0.5915 rank 87
    ep300: MAP 0.5148 rank 66
    ep400: MAP 0.4635 rank 61
    ep500: MAP 0.4302 rank 57
    ep600: MAP 0.4019 rank 58
    ep700: MAP 0.3863 rank 58
    ep800: MAP 0.3708 rank 59
    ep9

In [9]:
import os, torch, numpy as np
CKPT_DIR = HME_ROOT + '/results/dim_runs'

# distortion pairs: graph (BFS) distance vs embedded distance, for a sample of node pairs
from collections import defaultdict, deque
adj=defaultdict(list)
for u,v in edges_idx: adj[u].append(v); adj[v].append(u)
rng=np.random.default_rng(0); DPAIRS=[]
for s in rng.integers(0,N,size=2000):
    dd={int(s):0}; q=deque([int(s)])
    while q:
        x=q.popleft()
        for y in adj[x]:
            if y not in dd: dd[y]=dd[x]+1; q.append(y)
    t=int(rng.integers(0,N))
    if t in dd and t!=s: DPAIRS.append((int(s),t,dd[t]))

def d_poincare(a,allp):
    diff2=np.sum((allp-a)**2,axis=1); nu=1-np.sum(a**2); nv=1-np.sum(allp**2,axis=1)
    return np.arccosh(np.maximum(1+2*diff2/(nu*nv+1e-12),1.0))
def d_euclid(a,allp): return np.linalg.norm(allp-a,axis=1)

# SCALED distortion (De Sa comparable): fit best global scale, then mean relative error
def distortion_scaled(pos, dist_fn):
    emb_d=np.array([float(dist_fn(pos[u], pos[v:v+1])[0]) for (u,v,dg) in DPAIRS])
    grf_d=np.array([dg for (u,v,dg) in DPAIRS], dtype=float)
    c=np.dot(emb_d,grf_d)/(np.dot(emb_d,emb_d)+1e-12)
    return float(np.mean(np.abs(c*emb_d-grf_d)/grf_d))

print(f"{'method':<8}{'dim':>4}{'distortion':>12}")
for dim in [10,5,2]:
    for mode in ['graded','const','euclid','learned']:
        f=f'{CKPT_DIR}/{mode}_d{dim}.pt'
        if not os.path.exists(f):
            print(f'{mode:<8}{dim:>4}   (no checkpoint yet)'); continue
        ck=torch.load(f, map_location='cpu')
        if mode in ('euclid','learned'):
            pos=ck['emb'].numpy()                      # both save under 'emb'
            dfn = d_euclid if mode=='euclid' else d_poincare   # but learned uses Poincare dist!
        else:
            sd=ck['model']; key=[k for k in sd if 'embed' in k.lower()][0]
            pos=sd[key].numpy(); dfn=d_poincare
        if np.isnan(pos).any():
            print(f'{mode:<8}{dim:>4}   (NaN)'); continue
        print(f'{mode:<8}{dim:>4}{distortion_scaled(pos,dfn):>12.4f}')

method   dim  distortion
graded    10      0.1460
const     10      0.1109
euclid    10      0.1935
learned   10      0.1109
graded     5      0.1520
const      5      0.1161
euclid     5      0.2115
learned    5      0.1311
graded     2      0.2180
const      2      0.1639
euclid     2      0.3745
learned    2      0.1763


In [7]:
import os, torch, numpy as np
from collections import defaultdict, deque
CKPT_DIR = HME_ROOT + '/results/dim_runs'

# ---- distance functions ----
def d_poincare(a,allp):
    diff2=np.sum((allp-a)**2,axis=1); nu=1-np.sum(a**2); nv=1-np.sum(allp**2,axis=1)
    return np.arccosh(np.maximum(1+2*diff2/(nu*nv+1e-12),1.0))
def d_euclid(a,allp): return np.linalg.norm(allp-a,axis=1)

# ---- eval sets (fixed) ----
nbrs=defaultdict(set)
for u,v in edges_idx: nbrs[u].add(v); nbrs[v].add(u)
rng=np.random.default_rng(42)
EVAL=[edges_idx[i] for i in rng.choice(len(edges_idx),size=2000,replace=False)]

adj=defaultdict(list)
for u,v in edges_idx: adj[u].append(v); adj[v].append(u)
rng=np.random.default_rng(0); DPAIRS=[]
for s in rng.integers(0,N,size=2000):
    dd={int(s):0}; q=deque([int(s)])
    while q:
        x=q.popleft()
        for y in adj[x]:
            if y not in dd: dd[y]=dd[x]+1; q.append(y)
    t=int(rng.integers(0,N))
    if t in dd and t!=s: DPAIRS.append((int(s),t,dd[t]))

# ---- metrics ----
def eval_map_rank(pos, dist_fn):
    ranks=[]; aps=[]
    for (u,v) in EVAL:
        d=dist_fn(pos[u],pos); d[u]=np.inf; order=np.argsort(d)
        ranks.append(int(np.where(order==v)[0][0])+1)
        truth=nbrs[u]; hit=0; precs=[]
        for j,node in enumerate(order):
            if node in truth: hit+=1; precs.append(hit/(j+1))
            if hit==len(truth): break
        if precs: aps.append(np.mean(precs))
    return float(np.mean(aps)), float(np.mean(ranks))

def distortion_scaled(pos, dist_fn):
    emb_d=np.array([float(dist_fn(pos[u], pos[v:v+1])[0]) for (u,v,dg) in DPAIRS])
    grf_d=np.array([dg for (u,v,dg) in DPAIRS], dtype=float)
    c=np.dot(emb_d,grf_d)/(np.dot(emb_d,emb_d)+1e-12)
    return float(np.mean(np.abs(c*emb_d-grf_d)/grf_d))

# ---- load each checkpoint, compute all three metrics ----
def load_pos(mode, dim):
    f=f'{CKPT_DIR}/{mode}_d{dim}.pt'
    if not os.path.exists(f): return None, None
    ck=torch.load(f, map_location='cpu')
    if mode in ('euclid','learned'):
        pos=ck['emb'].numpy()
        dfn = d_euclid if mode=='euclid' else d_poincare
    else:
        sd=ck['model']; key=[k for k in sd if 'embed' in k.lower()][0]
        pos=sd[key].numpy(); dfn=d_poincare
    return pos, dfn

results={}
methods=['graded','const','euclid','learned']
for dim in [10,5,2]:
    for mode in methods:
        pos,dfn=load_pos(mode,dim)
        if pos is None or np.isnan(pos).any():
            results[(mode,dim)]=None; continue
        m,mr=eval_map_rank(pos,dfn)
        dist=distortion_scaled(pos,dfn)
        results[(mode,dim)]=(m,mr,dist)

# ---- print three tables ----
def cell(mode,dim,idx,fmt):
    r=results.get((mode,dim))
    return (fmt % r[idx]) if r else '   --  '

print("="*54)
print("MAP  (higher better)")
print(f"{'method':<9}{'d=10':>10}{'d=5':>10}{'d=2':>10}")
for mode in methods:
    print(f"{mode:<9}"+''.join(cell(mode,d,0,'%10.4f') for d in [10,5,2]))

print("\nMEAN RANK  (lower better, out of 46,816)")
print(f"{'method':<9}{'d=10':>10}{'d=5':>10}{'d=2':>10}")
for mode in methods:
    print(f"{mode:<9}"+''.join(cell(mode,d,1,'%10.0f') for d in [10,5,2]))

print("\nDISTORTION  (lower better, scaled)")
print(f"{'method':<9}{'d=10':>10}{'d=5':>10}{'d=2':>10}")
for mode in methods:
    print(f"{mode:<9}"+''.join(cell(mode,d,2,'%10.4f') for d in [10,5,2]))
print("="*54)

# save the full cube
import json
json.dump({f'{m}_d{d}':results[(m,d)] for m in methods for d in [10,5,2] if results.get((m,d))},
          open(f'{CKPT_DIR}/full_results.json','w'), indent=2)
print('saved full_results.json')

MAP  (higher better)
method         d=10       d=5       d=2
graded       0.8866    0.8288    0.5327
const        0.7752    0.7436    0.4248
euclid       0.8388    0.5797    0.1281
learned      0.7564    0.6764    0.6267

MEAN RANK  (lower better, out of 46,816)
method         d=10       d=5       d=2
graded          232       372       528
const           124       213      1056
euclid            5        12        94
learned          53       187       638

DISTORTION  (lower better, scaled)
method         d=10       d=5       d=2
graded       0.1460    0.1520    0.2180
const        0.1109    0.1161    0.1639
euclid       0.1935    0.2115    0.3745
learned      0.1109    0.1311    0.1763
saved full_results.json


In [8]:
# ============================================================
# LINE-INTEGRAL DISTANCE — the honest variable-metric version
# Integrates the modulated conformal factor along the chord u->v:
#     d(u,v) = integral  lambda_P(g(t)) * exp(alpha*kappa(g(t))) * ||g'(t)|| dt
# approximated with S=8 points. Accounts for the field ALONG the path,
# not just endpoints -> a real (approximate) metric, matching the paper.
# ============================================================

S_STEPS = 8

def line_integral_dist(model, u_idx, v_idx, kap, alpha, S=S_STEPS):
    u = model.embeddings[u_idx]; v = model.embeddings[v_idx]
    delta = v - u
    seg = delta.norm(dim=-1) / S
    t = (torch.arange(S, device=u.device) + 0.5) / S
    pts = u.unsqueeze(-2) + t.view(*([1]*(u.dim()-1)), S, 1) * delta.unsqueeze(-2)
    sq = (pts*pts).sum(-1).clamp(max=1-1e-5)
    base = 2.0 / (1.0 - sq)
    ku = kap[u_idx].unsqueeze(-1); kv = kap[v_idx].unsqueeze(-1)
    kap_path = ku + t.view(*([1]*(ku.dim()-1)), S) * (kv - ku)
    lam = base * torch.exp(alpha * kap_path)
    return lam.sum(-1) * seg

def run_lineint(mode, dim, alpha=-2.0, epochs=1500, lr=50, K=50, seed=0, log_every=100):
    torch.manual_seed(seed); np.random.seed(seed)
    model = PoincareEmbedding(N, dim, 0.001).to(device)
    opt = geoopt.optim.RiemannianSGD(model.parameters(), lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT] = 0.0
    kf = graded_kappa(1.0).detach()
    def kap(): return torch.zeros(N, device=device) if mode == 'const' else kf
    E = np.array(edges_idx)
    for ep in range(epochs):
        eff = lr*0.01 if ep < 10 else lr
        for pg in opt.param_groups: pg['lr'] = eff
        perm = np.random.permutation(len(E))
        for bs in range(0, len(E), 1024):
            k = kap()
            b = E[perm[bs:bs+1024]]; a = torch.tensor(b[:,0]).to(device); p = torch.tensor(b[:,1]).to(device)
            ng = torch.tensor(sample_negs(b[:,0], K)).to(device)
            dp = line_integral_dist(model, a, p, k, alpha)
            a_rep = a.unsqueeze(1).expand(-1, K)
            dn = line_integral_dist(model, a_rep, ng, k, alpha)
            nk = (dp + torch.logsumexp(-torch.cat([dp.unsqueeze(1), dn], 1), 1)).mean()
            an = model.embeddings[a].norm(dim=-1); pn = model.embeddings[p].norm(dim=-1)
            loss = nk + 1.0*torch.relu(an - pn + 0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT] = 0.0
            opt.step()
        if (ep+1) % log_every == 0:
            pos = model.embeddings.detach().cpu().numpy()
            if np.isnan(pos).any():
                print(f'    {mode} d{dim} NaN at ep{ep+1}'); break
            m, mr = evaluate(pos)
            print(f'    {mode} d{dim} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
    return evaluate(model.embeddings.detach().cpu().numpy())

print(f'LINE-INTEGRAL graded (S={S_STEPS} steps), 1500 epochs, d=10/5/2\n')
lineint_results = {}
for dim in [10, 5, 2]:
    print(f'=== graded d={dim} (line-integral) ===')
    m, mr = run_lineint('graded', dim, alpha=-2.0, epochs=1500, seed=0)
    lineint_results[dim] = (m, mr)
    print(f'  >> graded d={dim} line-integral: MAP {m:.4f} rank {mr:.0f}\n')

LINE-INTEGRAL graded (S=8 steps), 1500 epochs, d=10/5/2

=== graded d=10 (line-integral) ===
    graded d10 ep100: MAP 0.8087 rank 386
    graded d10 ep200: MAP 0.8353 rank 282
    graded d10 ep300: MAP 0.8535 rank 234
    graded d10 ep400: MAP 0.8610 rank 206
    graded d10 ep500: MAP 0.8670 rank 195
    graded d10 ep600: MAP 0.8704 rank 196
    graded d10 ep700: MAP 0.8751 rank 187
    graded d10 ep800: MAP 0.8768 rank 177
    graded d10 ep900: MAP 0.8801 rank 169
    graded d10 ep1000: MAP 0.8799 rank 169
    graded d10 ep1100: MAP 0.8854 rank 170
    graded d10 ep1200: MAP 0.8854 rank 166
    graded d10 ep1300: MAP 0.8872 rank 154
    graded d10 ep1400: MAP 0.8867 rank 156
    graded d10 ep1500: MAP 0.8890 rank 149
  >> graded d=10 line-integral: MAP 0.8890 rank 149

=== graded d=5 (line-integral) ===
    graded d5 ep100: MAP 0.7168 rank 626
    graded d5 ep200: MAP 0.7408 rank 492
    graded d5 ep300: MAP 0.7587 rank 425
    graded d5 ep400: MAP 0.7679 rank 379
    graded d5 ep500

In [10]:
# d=2 alpha sweep in LINE-INTEGRAL (find d=2's real alpha; fix the disclosed weakness)
print('graded d=2 alpha sweep, LINE-INTEGRAL, 1500 ep:\n')
for alpha in [-0.8, -1.0, -1.3]:
    torch.manual_seed(0); np.random.seed(0)
    model=PoincareEmbedding(N,2,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=50)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    E=np.array(edges_idx); best=0
    for ep in range(1500):
        eff=50*0.01 if ep<10 else 50
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],50)).to(device)
            dp=line_integral_dist(model, a, p, kf, alpha)
            a_rep=a.unsqueeze(1).expand(-1,50)
            dn=line_integral_dist(model, a_rep, ng, kf, alpha)
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=model.embeddings[a].norm(dim=-1); pn=model.embeddings[p].norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%300==0:
            pos=model.embeddings.detach().cpu().numpy()
            if np.isnan(pos).any(): print(f'    a={alpha} NaN ep{ep+1}'); break
            m,mr=evaluate(pos)
            if m>best: best=m
            print(f'    a={alpha} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
    print(f'  >> graded d=2 alpha={alpha} (line-int): peak MAP {best:.4f}\n')

graded d=2 alpha sweep, LINE-INTEGRAL, 1500 ep:

    a=-0.8 ep300: MAP 0.5337 rank 880
    a=-0.8 ep600: MAP 0.5107 rank 865
    a=-0.8 ep900: MAP 0.4947 rank 863
    a=-0.8 ep1200: MAP 0.4948 rank 863
    a=-0.8 ep1500: MAP 0.5002 rank 862
  >> graded d=2 alpha=-0.8 (line-int): peak MAP 0.5337

    a=-1.0 ep300: MAP 0.5306 rank 840
    a=-1.0 ep600: MAP 0.5250 rank 768
    a=-1.0 ep900: MAP 0.5166 rank 777
    a=-1.0 ep1200: MAP 0.5065 rank 766
    a=-1.0 ep1500: MAP 0.5059 rank 760
  >> graded d=2 alpha=-1.0 (line-int): peak MAP 0.5306

    a=-1.3 ep300: MAP 0.5159 rank 829
    a=-1.3 ep600: MAP 0.5367 rank 778
    a=-1.3 ep900: MAP 0.5373 rank 769
    a=-1.3 ep1200: MAP 0.5265 rank 740
    a=-1.3 ep1500: MAP 0.5238 rank 719
  >> graded d=2 alpha=-1.3 (line-int): peak MAP 0.5373



In [15]:
# multi-seed d=5: graded vs const, error bars on the key low-dim result
print('multi-seed d=5 (3 seeds, 3000 ep):\n')
for mode in ['graded','const']:
    ms=[]; rs=[]
    for seed in [0,1,2]:
        torch.manual_seed(seed); np.random.seed(seed)
        model=PoincareEmbedding(N,5,0.001).to(device)
        opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=50)
        with torch.no_grad(): model.embeddings.data[ROOT]=0.0
        kf=graded_kappa(1.0).detach()
        def kap(): return torch.zeros(N,device=device) if mode=='const' else kf
        E=np.array(edges_idx)
        for ep in range(3000):
            eff=50*0.01 if ep<10 else 50
            for pg in opt.param_groups: pg['lr']=eff
            perm=np.random.permutation(len(E))
            for bs in range(0,len(E),1024):
                k=kap(); b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
                ng=torch.tensor(sample_negs(b[:,0],50)).to(device)
                ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
                bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
                if mode=='const': dp,dn=bp,bn
                else:
                    dp=bp*torch.exp(-2.0*0.5*(k[a]+k[p])); dn=bn*torch.exp(-2.0*0.5*(k[a].unsqueeze(1)+k[ng]))
                nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
                an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
                loss=nk+1.0*torch.relu(an-pn+0.05).mean()
                opt.zero_grad(); loss.backward()
                if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
                opt.step()
        m,mr=evaluate(model.embeddings.detach().cpu().numpy()); ms.append(m); rs.append(mr)
        print(f'  {mode} d=5 seed{seed}: MAP {m:.4f}')
    print(f'  >> {mode} d=5: MAP {np.mean(ms):.4f} +/- {np.std(ms):.4f}\n')

multi-seed d=5 (3 seeds, 3000 ep):

  graded d=5 seed0: MAP 0.8328
  graded d=5 seed1: MAP 0.8323
  graded d=5 seed2: MAP 0.8366
  >> graded d=5: MAP 0.8339 +/- 0.0019

  const d=5 seed0: MAP 0.7407
  const d=5 seed1: MAP 0.7410
  const d=5 seed2: MAP 0.7384
  >> const d=5: MAP 0.7400 +/- 0.0012



#DAG

In [16]:
# ============================================================
# STAGE 1: Download Gene Ontology, parse the DAG, count multi-parent nodes
# The headline stat: what % of nodes have 2+ parents (Sarkar CANNOT handle these)
# ============================================================
!wget -q http://current.geneontology.org/ontology/go-basic.obo -O go-basic.obo
print('downloaded go-basic.obo')

from collections import defaultdict

go_name = {}                      # GO id -> name
go_parents = defaultdict(list)    # GO id -> list of parent GO ids (is_a)
cur = None
obsolete = set()

with open('go-basic.obo') as f:
    for line in f:
        line = line.strip()
        if line == '[Term]':
            cur = {'id': None, 'is_a': [], 'obsolete': False}
        elif line == '[Typedef]':
            cur = None   # stop reading (typedefs aren't terms)
        elif cur is not None:
            if line.startswith('id: GO:'):
                cur['id'] = line[4:]
            elif line.startswith('name:'):
                if cur['id']: go_name[cur['id']] = line[6:]
            elif line.startswith('is_obsolete: true'):
                cur['obsolete'] = True
            elif line.startswith('is_a:'):
                pid = line.split('!')[0].replace('is_a:', '').strip()
                cur['is_a'].append(pid)
            elif line == '':
                if cur['id'] and not cur['obsolete']:
                    go_parents[cur['id']] = cur['is_a']
                    if cur['obsolete']: obsolete.add(cur['id'])
                cur = None

# keep only non-obsolete terms
all_terms = [t for t in go_name if t not in obsolete]
# filter parents to existing terms
for t in list(go_parents):
    go_parents[t] = [p for p in go_parents[t] if p in go_name and p not in obsolete]

# ---- the key statistics ----
n_total = len(all_terms)
parent_counts = [len(go_parents[t]) for t in all_terms]
n_multi = sum(1 for c in parent_counts if c >= 2)
n_roots = sum(1 for c in parent_counts if c == 0)
n_edges = sum(parent_counts)

print(f'\n=== Gene Ontology DAG structure ===')
print(f'total terms:            {n_total}')
print(f'total is_a edges:       {n_edges}')
print(f'roots (0 parents):      {n_roots}')
print(f'nodes with 1 parent:    {sum(1 for c in parent_counts if c==1)}')
print(f'nodes with 2+ parents:  {n_multi}  ({100*n_multi/n_total:.1f}%)  <-- SARKAR CANNOT HANDLE THESE')
print(f'max parents on a node:  {max(parent_counts)}')

# show a few multi-parent examples
print(f'\nExample multi-parent nodes:')
shown=0
for t in all_terms:
    if len(go_parents[t])>=2:
        print(f'  {t} ({go_name[t][:40]}) has {len(go_parents[t])} parents:')
        for p in go_parents[t][:3]:
            print(f'      -> {p} ({go_name[p][:40]})')
        shown+=1
        if shown>=3: break

downloaded go-basic.obo

=== Gene Ontology DAG structure ===
total terms:            48329
total is_a edges:       57824
roots (0 parents):      10087
nodes with 1 parent:    23450
nodes with 2+ parents:  14792  (30.6%)  <-- SARKAR CANNOT HANDLE THESE
max parents on a node:  9

Example multi-parent nodes:
  GO:0000001 (mitochondrion inheritance) has 2 parents:
      -> GO:0048308 (organelle inheritance)
      -> GO:0048311 (mitochondrion distribution)
  GO:0000011 (vacuole inheritance) has 2 parents:
      -> GO:0007033 (vacuole organization)
      -> GO:0048308 (organelle inheritance)
  GO:0000014 (single-stranded DNA endonuclease activit) has 2 parents:
      -> GO:0004520 (DNA endonuclease activity)
      -> GO:0016788 (hydrolase activity, acting on ester bond)


In [17]:
# ============================================================
# STAGE 2: Convert GO DAG into training structures with MULTI-PARENT handling
# Key difference from ICD-10: a node can have multiple parents, so
#   - "edges" include ALL parent->child relationships (not just one per node)
#   - "depth" = shortest path from any root (a node has one depth via its shallowest parent)
#   - hierarchy loss must hold for ALL parents of a node
# ============================================================
from collections import deque

# build index
go_terms = [t for t in go_name if t not in obsolete]
go_to_idx = {t:i for i,t in enumerate(go_terms)}
N_go = len(go_terms)

# edges: (parent_idx, child_idx) for EVERY is_a relationship (multi-parent = multiple edges into a child)
go_edges = []
children_of = defaultdict(list)
parents_of = defaultdict(list)
for child in go_terms:
    for par in go_parents[child]:
        if par in go_to_idx:
            ci, pi = go_to_idx[child], go_to_idx[par]
            go_edges.append((pi, ci))          # parent -> child
            children_of[pi].append(ci)
            parents_of[ci].append(pi)

print(f'GO: {N_go} nodes, {len(go_edges)} parent-child edges')
print(f'  multi-parent children: {sum(1 for c in range(N_go) if len(parents_of[c])>=2)}')

# depth via BFS from all roots (multi-parent: depth = shortest path from any root)
roots = [go_to_idx[t] for t in go_terms if len(go_parents[t])==0]
depth = {r:0 for r in roots}
q = deque(roots)
while q:
    x = q.popleft()
    for c in children_of[x]:
        if c not in depth or depth[c] > depth[x]+1:
            if c not in depth: q.append(c)
            depth[c] = min(depth.get(c, 1e9), depth[x]+1)
for i in range(N_go):
    if i not in depth: depth[i]=0   # isolated safety

# branching factor per node (for the graded kappa law)
go_branching = np.array([len(children_of[i]) for i in range(N_go)])
go_depth = np.array([depth[i] for i in range(N_go)])

# graded kappa for GO: same law, (log(1+b))^2 normalized to [-1,1]
raw = np.log1p(go_branching)**2
go_kappa = 2*(raw - raw.min())/(raw.max()-raw.min()+1e-9) - 1
go_kappa = torch.tensor(go_kappa, dtype=torch.float32, device=device)

print(f'  depth range: {go_depth.min()}-{go_depth.max()}, max branching: {go_branching.max()}')
print(f'  roots: {len(roots)}  (multi-root DAG, unlike single-root ICD-10)')
print('Stage 2 complete: go_edges, go_kappa, parents_of ready for training')

GO: 48329 nodes, 57824 parent-child edges
  multi-parent children: 14792
  depth range: 0-13, max branching: 438
  roots: 10087  (multi-root DAG, unlike single-root ICD-10)
Stage 2 complete: go_edges, go_kappa, parents_of ready for training


In [19]:
import numpy as np, torch

raw = np.log1p(go_branching)**2

# ROBUST normalization: scale by the 95th percentile, not the max,
# so a few giant hubs (branching 438) don't squash everyone else to -1.
p95 = np.percentile(raw, 95)
raw_clipped = np.clip(raw, 0, p95)                       # cap the outlier hubs
go_kappa_np = 2*(raw_clipped - raw_clipped.min())/(raw_clipped.max()-raw_clipped.min()+1e-9) - 1
go_kappa = torch.tensor(go_kappa_np, dtype=torch.float32, device=device)

print('FIXED go_kappa distribution:', np.round(np.percentile(go_kappa_np, [0,25,50,75,100]), 3))
print('  fraction at floor (-1):', np.mean(go_kappa_np < -0.99).round(3))
print('  fraction at ceil (+1):', np.mean(go_kappa_np > 0.99).round(3))

FIXED go_kappa distribution: [-1.    -1.    -1.    -0.701  1.   ]
  fraction at floor (-1): 0.717
  fraction at ceil (+1): 0.056


In [21]:
import numpy as np, torch

# GO is mostly leaves (branching=0), so (log b)^2 collapses. Use a feature that
# actually varies across nodes. Depth varies across all nodes (0-13 here).
# Combine depth + subtree influence for a graded field that isn't floor-pinned.

# subtree size (how many descendants) - varies even among internal nodes
from collections import deque
subtree = np.ones(N_go)
# process in reverse-topological (leaves first) via depth
order = np.argsort(-go_depth)   # deepest first
for i in order:
    for p in parents_of[i]:
        subtree[p] += subtree[i]

# feature: blend of branching and subtree, both log-scaled
feat = np.log1p(go_branching) + 0.5*np.log1p(subtree)
p5, p95 = np.percentile(feat,5), np.percentile(feat,95)
feat = np.clip(feat, p5, p95)
go_kappa_np = 2*(feat-feat.min())/(feat.max()-feat.min()+1e-9) - 1
go_kappa = torch.tensor(go_kappa_np, dtype=torch.float32, device=device)
print('depth+subtree kappa dist:', np.round(np.percentile(go_kappa_np,[0,25,50,75,100]),3))
print('  fraction at floor:', np.mean(go_kappa_np<-0.99).round(3))

depth+subtree kappa dist: [-1.    -1.    -1.    -0.357  1.   ]
  fraction at floor: 0.717


In [23]:
# ============================================================
# STAGE 3: Train graded curvature on the GO DAG (multi-parent)
# Differences from ICD-10:
#   - NO single-root freeze (10,087 roots; can't anchor one point)
#   - hierarchy loss holds for ALL parents of each node
#   - same graded law, line-integral distance
# ============================================================

# sanity: kappa distribution (max branching 438 could compress normal nodes)
import numpy as np
print('go_kappa distribution:', np.percentile(go_kappa.cpu().numpy(), [0,25,50,75,100]))

# GO negative sampler
def go_sample_negs(anchors, K):
    return np.random.randint(0, N_go, size=(len(anchors), K))

# GO evaluation: rank each child's TRUE PARENT(S) among all nodes
# (multi-parent: a hit is ranking ANY true parent high; MAP over the parent set)
go_nbrs = defaultdict(set)
for (pi,ci) in go_edges:
    go_nbrs[ci].add(pi); go_nbrs[pi].add(ci)
rng = np.random.default_rng(42)
GO_EVAL = [go_edges[i] for i in rng.choice(len(go_edges), size=2000, replace=False)]

def go_evaluate(pos):
    # rank by plain poincare distance (embedding quality is what we test)
    def dpois(a, allp):
        diff2=np.sum((allp-a)**2,axis=1); nu=1-np.sum(a**2); nv=1-np.sum(allp**2,axis=1)
        return np.arccosh(np.maximum(1+2*diff2/(nu*nv+1e-12),1.0))
    ranks=[]; aps=[]
    for (pi,ci) in GO_EVAL:
        d=dpois(pos[ci],pos); d[ci]=np.inf; order=np.argsort(d)
        ranks.append(int(np.where(order==pi)[0][0])+1)
        truth=go_nbrs[ci]; hit=0; precs=[]
        for j,node in enumerate(order):
            if node in truth: hit+=1; precs.append(hit/(j+1))
            if hit==len(truth): break
        if precs: aps.append(np.mean(precs))
    return float(np.mean(aps)), float(np.mean(ranks))

def train_go(mode='graded', dim=10, alpha=-2.0, epochs=1500, lr=50, K=50, seed=0, log_every=100):
    torch.manual_seed(seed); np.random.seed(seed)
    emb = geoopt.ManifoldParameter(
        torch.empty(N_go, dim, device=device).uniform_(-0.001, 0.001),
        manifold=geoopt.PoincareBall(c=1.0))
    opt = geoopt.optim.RiemannianSGD([emb], lr=lr)
    manifold = geoopt.PoincareBall(c=1.0)
    kap = torch.zeros(N_go, device=device) if mode=='const' else go_kappa
    E = np.array(go_edges)   # (parent, child) pairs
    for ep in range(epochs):
        eff = lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm = np.random.permutation(len(E))
        for bs in range(0, len(E), 1024):
            b = E[perm[bs:bs+1024]]
            par = torch.tensor(b[:,0]).to(device); ch = torch.tensor(b[:,1]).to(device)
            ng = torch.tensor(go_sample_negs(b[:,1], K)).to(device)
            ep_ = emb[par]; ec = emb[ch]; en = emb[ng]
            bp = manifold.dist(ec, ep_); bn = manifold.dist(ec.unsqueeze(1), en)
            if mode=='const':
                dp, dn = bp, bn
            else:
                dp = bp*torch.exp(alpha*0.5*(kap[ch]+kap[par]))
                dn = bn*torch.exp(alpha*0.5*(kap[ch].unsqueeze(1)+kap[ng]))
            nk = (dp + torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            # hierarchy loss: child deeper (larger norm) than its parent — holds per edge,
            # so multi-parent nodes are pushed below ALL their parents automatically
            cn = ec.norm(dim=-1); pn = ep_.norm(dim=-1)
            loss = nk + 1.0*torch.relu(pn - cn + 0.05).mean()
            opt.zero_grad(); loss.backward(); opt.step()
        if (ep+1)%log_every==0:
            pos = emb.detach().cpu().numpy()
            if np.isnan(pos).any(): print(f'  NaN ep{ep+1}'); break
            m,mr = go_evaluate(pos)
            print(f'  {mode} GO ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
    return go_evaluate(emb.detach().cpu().numpy())

go_kappa distribution: [-1.         -1.         -1.         -0.35666832  1.        ]


In [24]:
# The DAG experiment's REAL point: embed a multi-parent DAG that Sarkar can't.
# graded's law degenerates on leaf-heavy GO (71% leaves), so we test whether
# the METHOD embeds the DAG at all, and report graded vs const honestly.

print('=== Embedding the GO DAG (30.6% multi-parent; Sarkar cannot process this) ===\n')
for mode in ['const','graded']:
    m, mr = train_go(mode, dim=10, epochs=1500)
    print(f'>> {mode} GO DAG: MAP {m:.4f} rank {mr:.0f}\n')

print('Headline: our method embeds all 48,329 GO nodes including the')
print('14,792 (30.6%) with multiple parents. Sarkar/De Sa construction')
print('cannot process multi-parent nodes and fails on this input.')

=== Embedding the GO DAG (30.6% multi-parent; Sarkar cannot process this) ===

  const GO ep100: MAP 0.4579 rank 117
  const GO ep200: MAP 0.4735 rank 89
  const GO ep300: MAP 0.4917 rank 87
  const GO ep400: MAP 0.5100 rank 87
  const GO ep500: MAP 0.5181 rank 86
  const GO ep600: MAP 0.5230 rank 87
  const GO ep700: MAP 0.5223 rank 87
  const GO ep800: MAP 0.5241 rank 87
  const GO ep900: MAP 0.5272 rank 87
  const GO ep1000: MAP 0.5270 rank 87
  const GO ep1100: MAP 0.5290 rank 86
  const GO ep1200: MAP 0.5282 rank 86
  const GO ep1300: MAP 0.5314 rank 86
  const GO ep1400: MAP 0.5315 rank 86
  const GO ep1500: MAP 0.5334 rank 86
>> const GO DAG: MAP 0.5334 rank 86

  graded GO ep100: MAP 0.3939 rank 10035
  graded GO ep200: MAP 0.3896 rank 10231
  graded GO ep300: MAP 0.3869 rank 10200
  graded GO ep400: MAP 0.3827 rank 9982
  graded GO ep500: MAP 0.3811 rank 9722
  graded GO ep600: MAP 0.3805 rank 9405
  graded GO ep700: MAP 0.3816 rank 9088
  graded GO ep800: MAP 0.3821 rank 8777

In [25]:
# graded on GO at GENTLE alpha (-0.3) — avoid blowing out the 71% leaves.
# At alpha=-0.3, leaf-leaf multiplier = exp(-0.3 * -1) = exp(0.3) = 1.35 (mild),
# vs the alpha=-2.0 case where it was exp(2)=7.4 (destroyed the embedding).
print('=== graded GO at alpha=-0.3 (gentle, fair test) ===\n')
m, mr = train_go('graded', dim=10, alpha=-0.3, epochs=1500)
print(f'\n>> graded GO (alpha=-0.3): MAP {m:.4f} rank {mr:.0f}')
print(f'   vs const GO: MAP 0.5334 rank 86')

=== graded GO at alpha=-0.3 (gentle, fair test) ===

  graded GO ep100: MAP 0.4763 rank 213
  graded GO ep200: MAP 0.4596 rank 160
  graded GO ep300: MAP 0.4568 rank 154
  graded GO ep400: MAP 0.4587 rank 151
  graded GO ep500: MAP 0.4669 rank 150
  graded GO ep600: MAP 0.4777 rank 149
  graded GO ep700: MAP 0.4864 rank 150
  graded GO ep800: MAP 0.4901 rank 149
  graded GO ep900: MAP 0.4927 rank 150
  graded GO ep1000: MAP 0.4946 rank 150
  graded GO ep1100: MAP 0.4954 rank 149
  graded GO ep1200: MAP 0.4962 rank 150
  graded GO ep1300: MAP 0.4970 rank 149
  graded GO ep1400: MAP 0.4998 rank 149
  graded GO ep1500: MAP 0.4996 rank 150

>> graded GO (alpha=-0.3): MAP 0.4996 rank 150
   vs const GO: MAP 0.5334 rank 86


In [26]:
# ============================================================
# LEARNED-FIELD ABLATIONS (reviewer Q6): is learned's instability fundamental,
# or just an underpowered/under-regularized field?
#   A) deeper MLP field (not just tanh over 2 features)
#   B) smoothness regularizer on kappa
#   C) early freeze (freeze field after it learns, then train embedding)
# ============================================================

def run_learned_ablation(variant, dim=10, alpha=-0.8, epochs=1500, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0

    if variant=='deeper_mlp':
        field=nn.Sequential(nn.Linear(2,16),nn.Tanh(),nn.Linear(16,16),nn.Tanh(),nn.Linear(16,1),nn.Tanh()).to(device)
        fparams=list(field.parameters())
    else:
        W=nn.Parameter(torch.zeros(2,1,device=device)); fparams=[W]; field=None
    fopt=torch.optim.Adam(fparams,lr=0.2,weight_decay=1e-4)

    def get_kappa():
        if variant=='deeper_mlp': return field(kappa_raw).squeeze(-1)
        return torch.tanh(kappa_raw@W).squeeze(-1)

    E=np.array(edges_idx); best=0; best_ep=0
    freeze_at = 200 if variant=='early_freeze' else 10**9
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        field_frozen = ep >= freeze_at
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            k=get_kappa()
            if field_frozen: k=k.detach()
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            dp=bp*torch.exp(alpha*0.5*(k[a]+k[p])); dn=bn*torch.exp(alpha*0.5*(k[a].unsqueeze(1)+k[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            if variant=='smoothness':
                loss=loss + 0.5*((k[a]-k[p])**2).mean()
            opt.zero_grad(); fopt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
            if not field_frozen: fopt.step()
        if (ep+1)%150==0:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            if m>best: best=m; best_ep=ep+1
            kv=get_kappa().detach().cpu().numpy()
            binary_frac=np.mean((np.abs(kv)>0.9))
            print(f'    {variant} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}  |field binary%={binary_frac:.2f}|')
    print(f'  >> {variant}: peak MAP {best:.4f} at ep{best_ep}  (graded law=0.89; plain learned peaked 0.758 then collapsed)\n')
    return best

print('LEARNED-FIELD RESCUE ABLATIONS\n')
for variant in ['deeper_mlp','smoothness','early_freeze']:
    print(f'=== {variant} ===')
    run_learned_ablation(variant)

LEARNED-FIELD RESCUE ABLATIONS

=== deeper_mlp ===
    deeper_mlp ep150: MAP 0.8590 rank 20  |field binary%=1.00|
    deeper_mlp ep300: MAP 0.8786 rank 11  |field binary%=1.00|
    deeper_mlp ep450: MAP 0.8856 rank 8  |field binary%=1.00|
    deeper_mlp ep600: MAP 0.8916 rank 8  |field binary%=1.00|
    deeper_mlp ep750: MAP 0.8945 rank 8  |field binary%=1.00|
    deeper_mlp ep900: MAP 0.8966 rank 7  |field binary%=1.00|
    deeper_mlp ep1050: MAP 0.8971 rank 7  |field binary%=1.00|
    deeper_mlp ep1200: MAP 0.8989 rank 7  |field binary%=1.00|
    deeper_mlp ep1350: MAP 0.8992 rank 7  |field binary%=1.00|
    deeper_mlp ep1500: MAP 0.8997 rank 7  |field binary%=1.00|
  >> deeper_mlp: peak MAP 0.8997 at ep1500  (graded law=0.89; plain learned peaked 0.758 then collapsed)

=== smoothness ===
    smoothness ep150: MAP 0.7928 rank 142  |field binary%=0.00|
    smoothness ep300: MAP 0.7936 rank 109  |field binary%=0.00|
    smoothness ep450: MAP 0.7919 rank 104  |field binary%=0.00|
    sm

In [28]:
import os
CKPT_DIR=HME_ROOT + '/results/dim_runs'; os.makedirs(CKPT_DIR,exist_ok=True)

def run_deeper_learned_lineint(dim, alpha=-0.8, epochs=3000, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    ckpt=f'{CKPT_DIR}/deeperlearned_lineint_d{dim}.pt'
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    field=nn.Sequential(nn.Linear(2,16),nn.Tanh(),nn.Linear(16,16),nn.Tanh(),nn.Linear(16,1),nn.Tanh()).to(device)
    fopt=torch.optim.Adam(field.parameters(),lr=0.2,weight_decay=1e-4)
    E=np.array(edges_idx); best=0; best_ep=0
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            k=field(kappa_raw).squeeze(-1)                 # live learned field (grad flows through line-integral)
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            # LINE-INTEGRAL distance (honest metric) instead of reweighting
            dp=line_integral_dist(model, a, p, k, alpha)                    # [B]
            a_rep=a.unsqueeze(1).expand(-1,K)
            dn=line_integral_dist(model, a_rep, ng, k, alpha)              # [B,K]
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=model.embeddings[a].norm(dim=-1); pn=model.embeddings[p].norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); fopt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step(); fopt.step()
        if (ep+1)%150==0:
            pos=model.embeddings.detach().cpu().numpy()
            if np.isnan(pos).any(): print(f'  d{dim} NaN ep{ep+1}'); break
            m,mr=evaluate(pos)
            kv=field(kappa_raw).squeeze(-1).detach().cpu().numpy(); binary=np.mean(np.abs(kv)>0.9)
            if m>best: best=m; best_ep=ep+1
            deg='  [DEGRADING]' if m<best-0.01 else ''
            print(f'  deeper_mlp LINE-INT d{dim} ep{ep+1}: MAP {m:.4f} rank {mr:.0f} binary%={binary:.2f}{deg}')
            torch.save({'emb':torch.tensor(pos),'ep':ep+1,'best':best}, ckpt)
    print(f'  >> deeper_mlp LINE-INT d{dim}: best MAP {best:.4f} at ep{best_ep}\n')
    return best

for dim in [10,5,2]:
    print(f'=== deeper_mlp LINE-INTEGRAL d={dim} (3000 ep) ===')
    run_deeper_learned_lineint(dim, epochs=3000)

=== deeper_mlp LINE-INTEGRAL d=10 (3000 ep) ===
  deeper_mlp LINE-INT d10 ep150: MAP 0.8613 rank 22 binary%=1.00
  deeper_mlp LINE-INT d10 ep300: MAP 0.8788 rank 12 binary%=1.00
  deeper_mlp LINE-INT d10 ep450: MAP 0.8843 rank 10 binary%=1.00
  deeper_mlp LINE-INT d10 ep600: MAP 0.8960 rank 10 binary%=1.00
  deeper_mlp LINE-INT d10 ep750: MAP 0.9005 rank 9 binary%=1.00
  deeper_mlp LINE-INT d10 ep900: MAP 0.9030 rank 8 binary%=1.00
  deeper_mlp LINE-INT d10 ep1050: MAP 0.9026 rank 8 binary%=1.00
  deeper_mlp LINE-INT d10 ep1200: MAP 0.9045 rank 8 binary%=1.00
  deeper_mlp LINE-INT d10 ep1350: MAP 0.9068 rank 8 binary%=1.00
  deeper_mlp LINE-INT d10 ep1500: MAP 0.9086 rank 8 binary%=1.00
  deeper_mlp LINE-INT d10 ep1650: MAP 0.9100 rank 8 binary%=1.00
  deeper_mlp LINE-INT d10 ep1800: MAP 0.9100 rank 8 binary%=1.00
  deeper_mlp LINE-INT d10 ep1950: MAP 0.9232 rank 8 binary%=1.00
  deeper_mlp LINE-INT d10 ep2100: MAP 0.9126 rank 7 binary%=1.00  [DEGRADING]
  deeper_mlp LINE-INT d10 ep225

In [34]:
import os
CKPT_DIR=HME_ROOT + '/results/dim_runs'; os.makedirs(CKPT_DIR,exist_ok=True)

def run_kernel_field(dim=10, alpha=-2.0, epochs=1500, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    ckpt=f'{CKPT_DIR}/kernelfield_d{dim}.pt'
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    E=np.array(edges_idx)
    start=0; best=0
    # resume if checkpoint exists
    if os.path.exists(ckpt):
        ck=torch.load(ckpt, map_location=device)
        model.embeddings.data=ck['emb'].to(device)
        opt.load_state_dict(ck['opt']); start=ck['ep']; best=ck.get('best',0)
        print(f'  resumed from ep{start}')
    ref_pos = model.embeddings.detach().clone(); cache, h = build_cache(ref_pos)
    for ep in range(start, epochs):
        if ep>0 and ep%RE_ANCHOR==0:
            ref_pos = model.embeddings.detach().clone(); cache, h = build_cache(ref_pos)
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            dp = kernel_line_integral(model, a, p, ref_pos, cache, h, alpha)
            a_rep = a.unsqueeze(1).expand(-1,K)
            dn = kernel_line_integral(model, a_rep, ng, ref_pos, cache, h, alpha)
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=model.embeddings[a].norm(dim=-1); pn=model.embeddings[p].norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%100==0:
            pos=model.embeddings.detach().cpu().numpy()
            if np.isnan(pos).any(): print(f'  NaN ep{ep+1}'); break
            m,mr=evaluate(pos)
            if m>best: best=m
            print(f'  kernel-field d{dim} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
            # SAVE checkpoint every 100 ep
            torch.save({'emb':model.embeddings.detach().cpu(),'opt':opt.state_dict(),
                        'ep':ep+1,'best':best}, ckpt)
    return evaluate(model.embeddings.detach().cpu().numpy())

print('POSITION-DEPENDENT kernel field, d=10, 1500 ep (checkpointed)\n')
m,mr = run_kernel_field(dim=10, alpha=-2.0, epochs=1500)
print(f'\n>> kernel field d=10: MAP {m:.4f} rank {mr:.0f}  (vs per-node reweight ~0.89)')

POSITION-DEPENDENT kernel field, d=10, 1500 ep (checkpointed)

  kernel-field d10 ep100: MAP 0.7949 rank 396
  kernel-field d10 ep200: MAP 0.7391 rank 72
  kernel-field d10 ep300: MAP 0.7716 rank 99
  kernel-field d10 ep400: MAP 0.7749 rank 83
  kernel-field d10 ep500: MAP 0.7801 rank 57
  kernel-field d10 ep600: MAP 0.7803 rank 69
  kernel-field d10 ep700: MAP 0.7878 rank 77
  kernel-field d10 ep800: MAP 0.7924 rank 60
  kernel-field d10 ep900: MAP 0.7960 rank 66
  kernel-field d10 ep1000: MAP 0.8000 rank 62
  kernel-field d10 ep1100: MAP 0.8069 rank 71
  kernel-field d10 ep1200: MAP 0.8081 rank 59
  kernel-field d10 ep1300: MAP 0.8112 rank 53
  kernel-field d10 ep1400: MAP 0.8147 rank 57
  kernel-field d10 ep1500: MAP 0.8228 rank 57

>> kernel field d=10: MAP 0.8228 rank 57  (vs per-node reweight ~0.89)


In [35]:
import os
CKPT_DIR=HME_ROOT + '/results/dim_runs'; os.makedirs(CKPT_DIR,exist_ok=True)

# kernel field (position-dependent kappa) but with ENDPOINT REWEIGHTING (no line-integral)
# kappa(x) evaluated only at the two endpoint nodes' positions, then reweight.
def kappa_at_nodes(node_idx, ref_pos, cache, h):
    """kappa at each node's OWN position via spatial kernel avg of nearby nodes [P]."""
    pos = ref_pos[node_idx]                              # [P,D] the node's position
    pool = cache[node_idx]                               # [P,M] nearby node indices
    pool_pos = ref_pos[pool]; pool_kap = KAPPA_NODE[pool]
    sq = ((pos.unsqueeze(1)-pool_pos)**2).sum(-1)        # [P,M]
    kk = min(K_NEIGH, pool.shape[1])
    ks, kj = sq.topk(kk, 1, largest=False)
    w = torch.exp(-ks/(h*h))
    kap = torch.gather(pool_kap, 1, kj)
    return (w*kap).sum(-1)/w.sum(-1).clamp(min=1e-9)     # [P]

def run_kernel_reweight(dim=10, alpha=-2.0, epochs=1500, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    ckpt=f'{CKPT_DIR}/kernelreweight_d{dim}.pt'
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    E=np.array(edges_idx); start=0; best=0
    if os.path.exists(ckpt):
        ck=torch.load(ckpt,map_location=device); model.embeddings.data=ck['emb'].to(device)
        opt.load_state_dict(ck['opt']); start=ck['ep']; best=ck.get('best',0)
        print(f'  resumed from ep{start}')
    ref_pos = model.embeddings.detach().clone(); cache,h = build_cache(ref_pos)
    for ep in range(start, epochs):
        if ep>0 and ep%RE_ANCHOR==0:
            ref_pos=model.embeddings.detach().clone(); cache,h=build_cache(ref_pos)
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            # kernel kappa evaluated at each node's position (position-dependent field, endpoint eval)
            ka = kappa_at_nodes(a, ref_pos, cache, h)
            kp = kappa_at_nodes(p, ref_pos, cache, h)
            kn = kappa_at_nodes(ng.reshape(-1), ref_pos, cache, h).reshape(ng.shape)
            dp = bp*torch.exp(alpha*0.5*(ka+kp))
            dn = bn*torch.exp(alpha*0.5*(ka.unsqueeze(1)+kn))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%100==0:
            pos=model.embeddings.detach().cpu().numpy()
            if np.isnan(pos).any(): print(f'  NaN ep{ep+1}'); break
            m,mr=evaluate(pos)
            if m>best: best=m
            print(f'  kernel-reweight d{dim} ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
            torch.save({'emb':model.embeddings.detach().cpu(),'opt':opt.state_dict(),'ep':ep+1,'best':best},ckpt)
    return evaluate(model.embeddings.detach().cpu().numpy())

print('KERNEL FIELD + REWEIGHTING (position-dependent field, endpoint distance), d=10\n')
m,mr = run_kernel_reweight(dim=10, alpha=-2.0, epochs=1500)
print(f'\n>> kernel-reweight d=10: MAP {m:.4f} rank {mr:.0f}')

KERNEL FIELD + REWEIGHTING (position-dependent field, endpoint distance), d=10

  kernel-reweight d10 ep100: MAP 0.8834 rank 5
  kernel-reweight d10 ep200: MAP 0.9099 rank 5
  kernel-reweight d10 ep300: MAP 0.9334 rank 4
  kernel-reweight d10 ep400: MAP 0.9465 rank 4
  kernel-reweight d10 ep500: MAP 0.9532 rank 4
  kernel-reweight d10 ep600: MAP 0.9588 rank 4
  kernel-reweight d10 ep700: MAP 0.9612 rank 4
  kernel-reweight d10 ep800: MAP 0.9649 rank 4
  kernel-reweight d10 ep900: MAP 0.9661 rank 4
  kernel-reweight d10 ep1000: MAP 0.9704 rank 4
  kernel-reweight d10 ep1100: MAP 0.9734 rank 4
  kernel-reweight d10 ep1200: MAP 0.9730 rank 4
  kernel-reweight d10 ep1300: MAP 0.9724 rank 4
  kernel-reweight d10 ep1400: MAP 0.9774 rank 4
  kernel-reweight d10 ep1500: MAP 0.9751 rank 4

>> kernel-reweight d=10: MAP 0.9751 rank 4


In [5]:
import os
CKPT_DIR=HME_ROOT + '/results/dim_runs'; os.makedirs(CKPT_DIR,exist_ok=True)
KAPPA_NODE = graded_kappa(1.0).detach()
K_NEIGH = 32

def build_cache(ref_pos, M=64):
    N_ = ref_pos.shape[0]
    cache = torch.empty(N_, M, dtype=torch.long, device=ref_pos.device)
    rsq = (ref_pos**2).sum(-1)
    for s in range(0, N_, 2048):
        q = ref_pos[s:s+2048]
        sq = ((q*q).sum(-1,keepdim=True) + rsq.unsqueeze(0) - 2*q@ref_pos.t()).clamp(min=0)
        cache[s:s+2048] = sq.topk(M, 1, largest=False)[1]
    samp = ref_pos[torch.randperm(N_, device=ref_pos.device)[:2048]]
    sq = ((samp*samp).sum(-1,keepdim=True) + rsq.unsqueeze(0) - 2*samp@ref_pos.t()).clamp(min=0)
    h = sq.topk(K_NEIGH+1,1,largest=False)[0][:,-1].sqrt().median().item()
    return cache, max(h, 1e-3)

def kappa_at_nodes(node_idx, ref_pos, cache, h):
    pos = ref_pos[node_idx]
    pool = cache[node_idx]; pool_pos = ref_pos[pool]; pool_kap = KAPPA_NODE[pool]
    sq = ((pos.unsqueeze(1)-pool_pos)**2).sum(-1)
    kk = min(K_NEIGH, pool.shape[1])
    ks, kj = sq.topk(kk, 1, largest=False)
    w = torch.exp(-ks/(h*h))
    kap = torch.gather(pool_kap, 1, kj)
    return (w*kap).sum(-1)/w.sum(-1).clamp(min=1e-9)

def run_kernel_reweight_FROZEN(dim=10, alpha=-2.0, epochs=1500, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    E=np.array(edges_idx); best=0
    # FROZEN ref_pos: built ONCE, never re-anchored
    ref_pos = model.embeddings.detach().clone(); cache,h = build_cache(ref_pos)
    for ep in range(epochs):
        # NO RE-ANCHOR BLOCK  <-- this is the test
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            ka = kappa_at_nodes(a, ref_pos, cache, h)
            kp = kappa_at_nodes(p, ref_pos, cache, h)
            kn = kappa_at_nodes(ng.reshape(-1), ref_pos, cache, h).reshape(ng.shape)
            dp = bp*torch.exp(alpha*0.5*(ka+kp))
            dn = bn*torch.exp(alpha*0.5*(ka.unsqueeze(1)+kn))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%100==0:
            pos=model.embeddings.detach().cpu().numpy()
            if np.isnan(pos).any(): print(f'  NaN ep{ep+1}'); break
            m,mr=evaluate(pos)
            if m>best: best=m
            print(f'  FROZEN kernel-reweight ep{ep+1}: MAP {m:.4f} rank {mr:.0f}')
    return best

print('LEAK TEST: kernel-reweight with FROZEN ref_pos (no re-anchoring)\n')
print('If this stays ~0.975 -> re-anchoring was NOT the cause')
print('If this drops to ~0.89 or lower -> re-anchoring feedback was inflating it\n')
b = run_kernel_reweight_FROZEN(dim=10, alpha=-2.0, epochs=1500)
print(f'\n>> FROZEN kernel-reweight best MAP: {b:.4f}  (leaky version was 0.975)')

LEAK TEST: kernel-reweight with FROZEN ref_pos (no re-anchoring)

If this stays ~0.975 -> re-anchoring was NOT the cause
If this drops to ~0.89 or lower -> re-anchoring feedback was inflating it

  FROZEN kernel-reweight ep100: MAP 0.8834 rank 5
  FROZEN kernel-reweight ep200: MAP 0.9159 rank 4
  FROZEN kernel-reweight ep300: MAP 0.9345 rank 4
  FROZEN kernel-reweight ep400: MAP 0.9476 rank 4
  FROZEN kernel-reweight ep500: MAP 0.9552 rank 4
  FROZEN kernel-reweight ep600: MAP 0.9602 rank 4
  FROZEN kernel-reweight ep700: MAP 0.9665 rank 4
  FROZEN kernel-reweight ep800: MAP 0.9665 rank 4
  FROZEN kernel-reweight ep900: MAP 0.9691 rank 4
  FROZEN kernel-reweight ep1000: MAP 0.9729 rank 3
  FROZEN kernel-reweight ep1100: MAP 0.9759 rank 4
  FROZEN kernel-reweight ep1200: MAP 0.9773 rank 3
  FROZEN kernel-reweight ep1300: MAP 0.9764 rank 4
  FROZEN kernel-reweight ep1400: MAP 0.9803 rank 4
  FROZEN kernel-reweight ep1500: MAP 0.9811 rank 4

>> FROZEN kernel-reweight best MAP: 0.9811  (le

In [7]:
# SELF-CONTAINED DIAGNOSTIC: why does frozen kernel-reweight hit 0.98/rank-3?
import numpy as np
from collections import defaultdict

KAPPA_NODE = graded_kappa(1.0).detach()
K_NEIGH = 32

def build_cache(ref_pos, M=64):
    N_ = ref_pos.shape[0]
    cache = torch.empty(N_, M, dtype=torch.long, device=ref_pos.device)
    rsq = (ref_pos**2).sum(-1)
    for s in range(0, N_, 2048):
        q = ref_pos[s:s+2048]
        sq = ((q*q).sum(-1,keepdim=True) + rsq.unsqueeze(0) - 2*q@ref_pos.t()).clamp(min=0)
        cache[s:s+2048] = sq.topk(M, 1, largest=False)[1]
    samp = ref_pos[torch.randperm(N_, device=ref_pos.device)[:2048]]
    sq = ((samp*samp).sum(-1,keepdim=True) + rsq.unsqueeze(0) - 2*samp@ref_pos.t()).clamp(min=0)
    h = sq.topk(K_NEIGH+1,1,largest=False)[0][:,-1].sqrt().median().item()
    return cache, max(h, 1e-3)

def kappa_at_nodes(node_idx, ref_pos, cache, h):
    pos = ref_pos[node_idx]
    pool = cache[node_idx]; pool_pos = ref_pos[pool]; pool_kap = KAPPA_NODE[pool]
    sq = ((pos.unsqueeze(1)-pool_pos)**2).sum(-1)
    kk = min(K_NEIGH, pool.shape[1])
    ks, kj = sq.topk(kk, 1, largest=False)
    w = torch.exp(-ks/(h*h))
    kap = torch.gather(pool_kap, 1, kj)
    return (w*kap).sum(-1)/w.sum(-1).clamp(min=1e-9)

torch.manual_seed(0); np.random.seed(0)
model=PoincareEmbedding(N,10,0.001).to(device)
with torch.no_grad(): model.embeddings.data[ROOT]=0.0
ref_pos=model.embeddings.detach().clone()
cache,h=build_cache(ref_pos)

# 1. kappa spread
allnodes=torch.arange(N,device=device); kv=[]
for s in range(0,N,4096):
    kv.append(kappa_at_nodes(allnodes[s:s+4096], ref_pos, cache, h).detach())
kv=torch.cat(kv).cpu().numpy()
print(f'kappa_at_nodes: min={kv.min():.3f} max={kv.max():.3f} mean={kv.mean():.3f} std={kv.std():.3f}')
print(f'  KAPPA_NODE raw: min={float(KAPPA_NODE.min()):.3f} max={float(KAPPA_NODE.max()):.3f} std={float(KAPPA_NODE.std()):.3f}')

# 2. cache leaking tree structure at random init?
nbrs_check=defaultdict(set)
for u,v in edges_idx: nbrs_check[u].add(v); nbrs_check[v].add(u)
leak=0; total=0
for i in range(0,N,5000):
    cached=set(cache[i].cpu().numpy().tolist()); true=nbrs_check[i]
    if true: leak+=len(cached&true)/len(true); total+=1
print(f'\ncache-vs-true-neighbor overlap at INIT: {leak/max(total,1):.3f}  (should be ~0)')

# 3. CRITICAL: evaluate on RANDOM init - should be near-zero MAP
m0,mr0=evaluate(model.embeddings.detach().cpu().numpy())
print(f'\nevaluate on RANDOM init: MAP {m0:.4f} rank {mr0:.0f}  (should be ~0 / rank ~23000)')

kappa_at_nodes: min=-1.000 max=-0.712 mean=-0.903 std=0.038
  KAPPA_NODE raw: min=-1.000 max=1.000 std=0.208

cache-vs-true-neighbor overlap at INIT: 0.000  (should be ~0)

evaluate on RANDOM init: MAP 0.0003 rank 23309  (should be ~0 / rank ~23000)


In [10]:
# ============================================================
# HARD-CODED BINARY kappa (reviewer Q6): does a hand-assigned leaf/internal
# curvature match the LEARNED field's binary result (0.92 at d=10)?
# If yes -> you don't need to LEARN it; the binary rule IS the finding.
#
# The learned field collapsed to binary. So we test: assign leaves one kappa,
# internal nodes another, hard-coded. Sweep the two levels to find the best.
# ============================================================

# is each node a leaf (0 children) or internal?
child_count = np.zeros(N, dtype=int)
for (p, c) in edges_idx:   # (parent, child)
    child_count[p] += 1
is_leaf = (child_count == 0)
print(f'leaves: {is_leaf.sum()} ({100*is_leaf.mean():.1f}%), internal: {(~is_leaf).sum()}')

def make_binary_kappa(leaf_val, internal_val):
    k = np.where(is_leaf, leaf_val, internal_val).astype(np.float32)
    return torch.tensor(k, device=device)

def run_binary_kappa(leaf_val, internal_val, dim=10, alpha=-2.0, epochs=1500, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf = make_binary_kappa(leaf_val, internal_val)
    E=np.array(edges_idx); best=0
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            dp=bp*torch.exp(alpha*0.5*(kf[a]+kf[p])); dn=bn*torch.exp(alpha*0.5*(kf[a].unsqueeze(1)+kf[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%300==0:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            if m>best: best=m
    return best

# sweep the two binary levels (leaf_val, internal_val) at d=10
print('\nHard-coded binary kappa sweep, d=10, 1500ep')
print('(learned field found binary and hit 0.92; can we match it by hand?)\n')
configs = [
    (-1.0, +1.0),   # leaves low, internal high (matches graded law direction)
    (+1.0, -1.0),   # opposite
    (-1.0,  0.0),   # leaves low, internal neutral
    ( 0.0, +1.0),   # leaves neutral, internal high
    (-1.0, +0.5),
]
for lv, iv in configs:
    b = run_binary_kappa(lv, iv, dim=10, alpha=-2.0, epochs=1500)
    print(f'  leaf={lv:+.1f} internal={iv:+.1f}: MAP {b:.4f}')
print('\n  compare: learned binary field 0.92, graded (log b)^2 0.89, const 0.775')

leaves: 36042 (77.0%), internal: 10775

Hard-coded binary kappa sweep, d=10, 1500ep
(learned field found binary and hit 0.92; can we match it by hand?)

  leaf=-1.0 internal=+1.0: MAP 0.6454
  leaf=+1.0 internal=-1.0: MAP 0.3896


KeyboardInterrupt: 

In [11]:
# ============================================================
# HARD-CODED BINARY kappa (reviewer Q6): does a hand-assigned leaf/internal
# curvature match the LEARNED field's binary result (0.92 at d=10)?
# If yes -> you don't need to LEARN it; the binary rule IS the finding.
#
# The learned field collapsed to binary. So we test: assign leaves one kappa,
# internal nodes another, hard-coded. Sweep the two levels to find the best.
# ============================================================

# is each node a leaf (0 children) or internal?
child_count = np.zeros(N, dtype=int)
for (p, c) in edges_idx:   # (parent, child)
    child_count[p] += 1
is_leaf = (child_count == 0)
print(f'leaves: {is_leaf.sum()} ({100*is_leaf.mean():.1f}%), internal: {(~is_leaf).sum()}')

def make_binary_kappa(leaf_val, internal_val):
    k = np.where(is_leaf, leaf_val, internal_val).astype(np.float32)
    return torch.tensor(k, device=device)

def run_binary_kappa(leaf_val, internal_val, dim=10, alpha=-2.0, epochs=1500, lr=50, K=50, seed=0):
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf = make_binary_kappa(leaf_val, internal_val)
    E=np.array(edges_idx); best=0
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            dp=bp*torch.exp(alpha*0.5*(kf[a]+kf[p])); dn=bn*torch.exp(alpha*0.5*(kf[a].unsqueeze(1)+kf[ng]))
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%300==0:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            if m>best: best=m
    return best

# FAIR binary test: gentler alpha (so binary isn't destructive) + wider level sweep
# 77% leaves means leaf-leaf multiplier dominates; at alpha=-2 with +-1 binary that's
# exp(2)=7.4x on most pairs = destructive. Test gentler alpha and smaller gaps.
print('Fair hard-coded binary sweep (gentler alpha, smaller gaps):\n')
for alpha in [-0.8, -0.3]:
    for lv, iv in [(-1.0,+1.0), (-0.5,+0.5), (-0.3,+0.3), (0.0,+1.0), (-1.0,0.0)]:
        b = run_binary_kappa(lv, iv, dim=10, alpha=alpha, epochs=1000)
        print(f'  alpha={alpha} leaf={lv:+.1f} int={iv:+.1f}: MAP {b:.4f}')
    print()
print('  targets: learned 0.92, graded 0.89, const 0.775')

leaves: 36042 (77.0%), internal: 10775
Fair hard-coded binary sweep (gentler alpha, smaller gaps):

  alpha=-0.8 leaf=-1.0 int=+1.0: MAP 0.6674
  alpha=-0.8 leaf=-0.5 int=+0.5: MAP 0.6914
  alpha=-0.8 leaf=-0.3 int=+0.3: MAP 0.7266
  alpha=-0.8 leaf=+0.0 int=+1.0: MAP 0.6743
  alpha=-0.8 leaf=-1.0 int=+0.0: MAP 0.7487

  alpha=-0.3 leaf=-1.0 int=+1.0: MAP 0.7146
  alpha=-0.3 leaf=-0.5 int=+0.5: MAP 0.7442
  alpha=-0.3 leaf=-0.3 int=+0.3: MAP 0.7574
  alpha=-0.3 leaf=+0.0 int=+1.0: MAP 0.7255
  alpha=-0.3 leaf=-1.0 int=+0.0: MAP 0.7672

  targets: learned 0.92, graded 0.89, const 0.775


In [12]:
# What did the learned field's binary ACTUALLY look like? Load a learned checkpoint
# and inspect: what are the two kappa values, and does the split = leaf/internal?
import os, torch
# retrain a quick learned field and inspect its kappa (since checkpoints may be gone)
torch.manual_seed(0); np.random.seed(0)
model=PoincareEmbedding(N,10,0.001).to(device)
opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=50)
with torch.no_grad(): model.embeddings.data[ROOT]=0.0
field=nn.Sequential(nn.Linear(2,16),nn.Tanh(),nn.Linear(16,16),nn.Tanh(),nn.Linear(16,1),nn.Tanh()).to(device)
fopt=torch.optim.Adam(field.parameters(),lr=0.2,weight_decay=1e-4)
E=np.array(edges_idx)
for ep in range(600):  # enough to collapse to binary
    eff=50*0.01 if ep<10 else 50
    for pg in opt.param_groups: pg['lr']=eff
    perm=np.random.permutation(len(E))
    for bs in range(0,len(E),1024):
        k=field(kappa_raw).squeeze(-1)
        b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
        ng=torch.tensor(sample_negs(b[:,0],50)).to(device)
        ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
        bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
        dp=bp*torch.exp(-0.8*0.5*(k[a]+k[p])); dn=bn*torch.exp(-0.8*0.5*(k[a].unsqueeze(1)+k[ng]))
        nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
        an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
        loss=nk+1.0*torch.relu(an-pn+0.05).mean()
        opt.zero_grad(); fopt.zero_grad(); loss.backward()
        if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
        opt.step(); fopt.step()
kv = field(kappa_raw).squeeze(-1).detach().cpu().numpy()
# what are the two clusters?
print(f'learned kappa: unique-ish values, min={kv.min():.3f} max={kv.max():.3f}')
print(f'  leaf nodes:     kappa mean={kv[is_leaf].mean():.3f} std={kv[is_leaf].std():.3f}')
print(f'  internal nodes: kappa mean={kv[~is_leaf].mean():.3f} std={kv[~is_leaf].std():.3f}')
# is the split leaf/internal, or something else (e.g. by branching factor threshold)?
b_arr = child_count
for thresh in [0,1,2,5,10]:
    grp = b_arr > thresh
    # how well does "branching > thresh" predict kappa sign?
    from scipy.stats import pointbiserialr
    print(f'  corr(kappa, branching>{thresh}): {np.corrcoef(kv, grp.astype(float))[0,1]:.3f}')

learned kappa: unique-ish values, min=-1.000 max=-1.000
  leaf nodes:     kappa mean=-1.000 std=0.000
  internal nodes: kappa mean=-1.000 std=0.000
  corr(kappa, branching>0): -0.957
  corr(kappa, branching>1): -0.952
  corr(kappa, branching>2): -0.922
  corr(kappa, branching>5): -0.645
  corr(kappa, branching>10): -0.179


In [5]:
# ============================================================
# TEMPERATURE TEST: was the learned field's 0.92 just a global loss-temperature
# rescale, not real curvature?
#
# The learned field collapsed to kappa = -1.0 for EVERY node (constant, not binary).
# With alpha=-0.8, that's a uniform multiplier exp(-0.8 * 0.5 * (-1 + -1)) = exp(0.8)
# = 2.23x applied to ALL distances equally. A global constant multiplier on all
# distances just sharpens the softmax ranking loss - a "temperature" effect that
# encodes NO per-node curvature structure.
#
# TEST: does plain constant curvature, with the same global x-scale, also hit 0.92?
# If yes -> the learned field "learned" nothing but a scalar; curvature did nothing.
# We also scale GRADED to check whether it benefits from temperature too (fair comparison).
# ============================================================

def run_scaled(mode, scale, dim=10, alpha=-2.0, epochs=1500, lr=50, K=50, seed=0):
    """mode: 'const' (uniform x scale) or 'graded' (per-node kappa THEN x scale)."""
    torch.manual_seed(seed); np.random.seed(seed)
    model=PoincareEmbedding(N,dim,0.001).to(device)
    opt=geoopt.optim.RiemannianSGD(model.parameters(),lr=lr)
    with torch.no_grad(): model.embeddings.data[ROOT]=0.0
    kf=graded_kappa(1.0).detach()
    E=np.array(edges_idx); best=0
    for ep in range(epochs):
        eff=lr*0.01 if ep<10 else lr
        for pg in opt.param_groups: pg['lr']=eff
        perm=np.random.permutation(len(E))
        for bs in range(0,len(E),1024):
            b=E[perm[bs:bs+1024]]; a=torch.tensor(b[:,0]).to(device); p=torch.tensor(b[:,1]).to(device)
            ng=torch.tensor(sample_negs(b[:,0],K)).to(device)
            ea=model.embeddings[a]; ep_=model.embeddings[p]; en=model.embeddings[ng]
            bp=model.manifold.dist(ea,ep_); bn=model.manifold.dist(ea.unsqueeze(1),en)
            if mode=='const':
                dp=bp*scale; dn=bn*scale                      # pure global temperature
            else:  # graded, THEN global scale
                dp=bp*torch.exp(alpha*0.5*(kf[a]+kf[p]))*scale
                dn=bn*torch.exp(alpha*0.5*(kf[a].unsqueeze(1)+kf[ng]))*scale
            nk=(dp+torch.logsumexp(-torch.cat([dp.unsqueeze(1),dn],1),1)).mean()
            an=ea.norm(dim=-1); pn=ep_.norm(dim=-1)
            loss=nk+1.0*torch.relu(an-pn+0.05).mean()
            opt.zero_grad(); loss.backward()
            if model.embeddings.grad is not None: model.embeddings.grad[ROOT]=0.0
            opt.step()
        if (ep+1)%300==0:
            m,mr=evaluate(model.embeddings.detach().cpu().numpy())
            if m>best: best=m
    return best

print('TEMPERATURE TEST — is the learned 0.92 just a global loss-scale?\n')
print('CONSTANT curvature x global scale (no per-node structure):')
for scale in [1.0, 2.23, 3.0, 5.0, 8.0]:
    b=run_scaled('const', scale, epochs=1500)
    print(f'  const x{scale:.2f}: MAP {b:.4f}')

print('\nGRADED (per-node kappa) x global scale (does it ALSO benefit from temperature?):')
for scale in [1.0, 2.23, 5.0]:
    b=run_scaled('graded', scale, alpha=-2.0, epochs=1500)
    print(f'  graded x{scale:.2f}: MAP {b:.4f}')

print('\n  learned field (constant kappa=-1) hit 0.92')
print('  if const x2.23 ~ 0.92 -> learned "curvature" was pure temperature, contributed nothing')
print('  if graded also jumps with scale -> temperature is a separate knob; compare at MATCHED scale')

TEMPERATURE TEST — is the learned 0.92 just a global loss-scale?

CONSTANT curvature x global scale (no per-node structure):
  const x1.00: MAP 0.7736
  const x2.23: MAP 0.9007
  const x3.00: MAP 0.9410
  const x5.00: MAP 0.9774
  const x8.00: MAP 0.9913

GRADED (per-node kappa) x global scale (does it ALSO benefit from temperature?):
  graded x1.00: MAP 0.8810
  graded x2.23: MAP 0.9068
  graded x5.00: MAP 0.7418

  learned field (constant kappa=-1) hit 0.92
  if const x2.23 ~ 0.92 -> learned "curvature" was pure temperature, contributed nothing
  if graded also jumps with scale -> temperature is a separate knob; compare at MATCHED scale
